# TopoEyeNet v2: Bidirectional Topology-Guided Retinal Vessel Segmentation

**A clean rewrite of the TopoEyeNet framework**, redesigned around three tightly-integrated,
validated novel modules rather than the full 9-module wishlist. This follows the
project's own stated principle: *"2–3 well-integrated innovations that are thoroughly
validated is generally stronger than 8–10 loosely connected modules."*

## What changed vs. the original notebook

The original notebook's actual model (as opposed to its docstring) was a **single shared
decoder with two independent 1×1-conv heads** — mask and skeleton never interacted except
through the loss function. This rewrite makes three concrete architectural changes:

| # | Module | Replaces / addresses |
|---|---|---|
| 1 | **BTGI** — Bidirectional Topology-Guided Interaction | Gated cross-branch fusion between mask/skeleton decoders at 3 decoder stages (not concatenation) |
| 2 | **ATCL** — Adaptive Topology Consistency Loss | Per-pixel loss weight from local vessel radius + junction density, replacing fixed loss coefficients |
| 3 | **JAA** — Junction-Aware Attention | Attention map learned from a weakly-supervised junction/bifurcation head, applied to mask features |

Graph-based constraints and a dedicated connectivity-refinement block (prompt items #5, #8)
were deliberately **left out** — they would roughly double parameter count for a benefit the
lighter ATCL + JAA combination already targets, and the prompt itself flags "lightweight
clinical deployment" as a requirement.

## Practical fixes made along the way

- **Unified pipeline.** The original notebook had two disconnected pipelines (a whole-image
  resize pipeline reporting 19% mask Dice, and a patch-based native-resolution pipeline
  reporting 83.7% Dice) evaluated as if comparable. This rewrite uses **one pipeline** —
  native-resolution patches — throughout, since resizing was destroying thin vessels before
  training even started (measured skeleton retention ≈27.6% after resize in the original run).
- Every ablation stage (baseline → +skeleton head → +BTGI → +ATCL → +JAA → full) is a
  **single model class with flags**, not five different classes, so the ablation table is
  guaranteed to isolate one change at a time.
- All new modules were unit-tested for shape/gradient correctness before being wired in
  (see the assert cells inline).

## Honest scope note

This notebook is written to **run on Kaggle** against the paths already in `CFG`
(RetinoMix-5, DRIVE, EyePACS) with a GPU. It was **built and syntax/shape-tested in a
CPU sandbox without your datasets**, so architecture, loss, and metric logic are verified,
but end-to-end numbers, baseline comparisons, and the full 10-seed statistical protocol
need to be run by you on Kaggle. Cells that need real data are marked; everything else
executes as-is. Treat this as a v2 foundation you run and iterate on, not a finished
paper's worth of results.


In [ ]:
# ============================================================
# CELL 0: ENVIRONMENT INSTALL (pinned, Kaggle-tested versions)
# ============================================================
# Same pinned stack as the original notebook, with scikit-posthocs added
# for the Nemenyi post-hoc / critical-difference diagram in Cell 13.
#
# IMPORTANT -- "Run All" behavior: this cell restarts the kernel via
# os._exit(0) after installing, which means everything below it will NOT
# execute in the same "Run All" pass (the kernel dies mid-run). This cell
# is now IDEMPOTENT: if the pinned versions are already installed (i.e.
# this is your second "Run All"), it skips straight through with no
# install and no restart, so:
#   1st Run All -> installs, restarts kernel, stops here
#   2nd Run All -> detects correct versions, skips this cell instantly,
#                  continues straight through to the main model
# If training doesn't seem to happen, check whether this cell just
# restarted the kernel on you -- that is the #1 cause of "nothing after
# Cell 0 ran."

import subprocess, os, time


def _versions_already_correct():
    try:
        import numpy, scipy, torch, segmentation_models_pytorch as smp
        return (numpy.__version__ == "2.1.3" and scipy.__version__ == "1.14.1"
                and torch.__version__.startswith("2.3.0") and smp.__version__ == "0.3.3")
    except Exception:
        return False


if _versions_already_correct():
    print("Pinned environment already installed -- skipping install/restart, "
          "continuing to Cell 1.")
else:
    def run(cmd):
        print(f"$ {cmd}")
        subprocess.run(cmd, shell=True, check=False)

    # Core ABI pair
    run("pip uninstall -y -q numpy scipy")
    run("pip install -q numpy==2.1.3 scipy==1.14.1")

    # PyTorch
    run("pip install -q --no-deps torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu118")

    # Vision stack
    run("pip install -q timm==0.9.2")
    run("pip install -q segmentation-models-pytorch==0.3.3 --no-deps")
    run("pip install -q albucore==0.0.16 albumentations==1.4.11 --no-deps")
    run("pip install -q scikit-image==0.24.0 --no-deps")

    # Utils
    run("pip install -q networkx==3.3 grad-cam plotly==5.22.0 kaleido==0.2.1 wandb --no-deps")
    run("pip install -q scikit-posthocs statsmodels tabulate")  # NEW: stats + LaTeX/markdown tables

    # SMP deps + grad-cam dep
    run("pip install -q munch pretrainedmodels==0.7.4 efficientnet-pytorch==0.7.1 ttach --no-deps")

    # Lock ABI
    run("pip install -q --force-reinstall --no-deps numpy==2.1.3 scipy==1.14.1")

    print("\nInstall complete. Restarting kernel...")
    time.sleep(2)
    os._exit(0)


$ pip uninstall -y -q numpy scipy
$ pip install -q numpy==2.1.3 scipy==1.14.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 49.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.1.3 which is incompatible.


$ pip install -q --no-deps torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.6/839.6 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 18.2 MB/s eta 0:00:00
$ pip install -q timm==0.9.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 35.4 MB

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.3.0+cu118 which is incompatible.


$ pip install -q segmentation-models-pytorch==0.3.3 --no-deps
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 4.1 MB/s eta 0:00:00
$ pip install -q albucore==0.0.16 albumentations==1.4.11 --no-deps
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 6.2 MB/s eta 0:00:00
$ pip install -q scikit-image==0.24.0 --no-deps
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 82.2 MB/s eta 0:00:00
$ pip install -q networkx==3.3 grad-cam plotly==5.22.0 kaleido==0.2.1 wandb --no-deps
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 24.6 MB/s eta 0:00:00
$ pip install -q scikit-posthocs statsmodels tabulate
$ pip install -q munch pretrainedmodels==0.7.4 efficientnet-pytorch==0.7.1 ttach --no-deps
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# ============================================================
# CELL 1: ENVIRONMENT SETUP
# ============================================================
import sys
print(f"Python: {sys.version}")

import numpy as np, scipy
print(f"NumPy: {np.__version__}  |  SciPy: {scipy.__version__}")

import torch, torchvision
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

import segmentation_models_pytorch as smp
from segmentation_models_pytorch.decoders.unet.decoder import UnetDecoder
print(f"SMP: {smp.__version__}")

from skimage import morphology, measure
from skimage.morphology import skeletonize, remove_small_objects
from scipy import ndimage
from scipy.ndimage import distance_transform_edt, convolve
print("scikit-image + scipy.ndimage: OK")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx
print("NetworkX + Matplotlib: OK")

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import SemanticSegmentationTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
    HAS_GRADCAM = True
    print("grad-cam: OK")
except ImportError:
    HAS_GRADCAM = False
    print("grad-cam: NOT INSTALLED -- run `pip install grad-cam` for the XAI cells")

try:
    from scipy.stats import shapiro, wilcoxon, ttest_rel, friedmanchisquare
    print("scipy.stats: OK")
except ImportError:
    raise

try:
    import scikit_posthocs as sp
    HAS_POSTHOCS = True
    print("scikit-posthocs: OK")
except ImportError:
    HAS_POSTHOCS = False
    print("scikit-posthocs: NOT INSTALLED -- run `pip install scikit-posthocs` for Nemenyi CD diagrams")

import os, re, json, time, random, warnings, gc, math
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Union, Callable
import pandas as pd
from PIL import Image
import cv2
cv2.setNumThreads(0)  # avoids deadlocks with DataLoader worker processes
warnings.filterwarnings('ignore')

# ---- device / multi-GPU ----
device_count = torch.cuda.device_count()
print(f"\nDetected {device_count} GPU(s)")
for i in range(device_count):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | {p.total_memory / 1e9:.1f} GB")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Primary device: {device}")


def wrap_dual_gpu(model):
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
    return model.to(device)


def clear_gpu_cache():
    gc.collect()
    for i in range(torch.cuda.device_count()):
        torch.cuda.empty_cache()
        torch.cuda.synchronize(device=i)


def get_gpu_memory():
    for i in range(torch.cuda.device_count()):
        a = torch.cuda.memory_allocated(i) / 1e9
        r = torch.cuda.memory_reserved(i) / 1e9
        print(f"  GPU {i}: Allocated {a:.2f} GB | Reserved {r:.2f} GB")


def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)


print("\nCELL 1 COMPLETE")


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy: 2.1.3  |  SciPy: 1.14.1
PyTorch: 2.3.0+cu118  |  CUDA: True
SMP: 0.3.3
scikit-image + scipy.ndimage: OK
NetworkX + Matplotlib: OK
grad-cam: OK
scipy.stats: OK
scikit-posthocs: OK

Detected 2 GPU(s)
  GPU 0: Tesla T4 | 15.6 GB
  GPU 1: Tesla T4 | 15.6 GB
Primary device: cuda:0

CELL 1 COMPLETE


In [2]:
# ============================================================
# CELL 2: UNIFIED CONFIGURATION
# ============================================================
# One config, one pipeline (native-resolution patches). The whole-image
# resize pipeline from the original notebook is dropped: at IMG_SIZE=512
# it measured ~27.6% skeleton-pixel retention after resize on this data,
# i.e. it was throwing away most thin vessels before training even began.
# Patches are cut from the native-resolution image/mask, so no resize of
# ground truth ever happens.

from dataclasses import dataclass, field
from typing import List, Tuple, Optional


@dataclass
class CFG:
    EXPERIMENT_NAME: str = "topoeyenet_v2"
    SEED: int = 42
    SEEDS: Tuple[int, ...] = (42, 43, 44)  # multi-seed protocol for statistical validation

    # ---- data sources (unchanged paths from the original notebook) ----
    RETINOMIX_IMAGE_DIR: str = "/kaggle/input/datasets/parikshakaur/retinomix-5/RETINOMIX-5/images"
    RETINOMIX_MASK_DIR: str = "/kaggle/input/datasets/parikshakaur/retinomix-5/RETINOMIX-5/mask"

    DRIVE_TRAIN_IMAGE_DIR: str = "/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/training/images"
    DRIVE_TRAIN_MASK_DIR: str = "/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/training/1st_manual"
    DRIVE_TEST_IMAGE_DIR: str = "/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/test/images"
    DRIVE_TEST_MASK_DIR: str = "/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/test/mask"

    # ---- output dirs ----
    OUTPUT_DIR: str = "/kaggle/working/output"
    CHECKPOINT_DIR: str = "/kaggle/working/checkpoints"
    LOG_DIR: str = "/kaggle/working/logs"
    PRECOMPUTED_DIR: str = "/kaggle/working/precomputed"
    FIGURES_DIR: str = "/kaggle/working/figures"

    # ---- patch pipeline ----
    PATCH_SIZE: int = 128
    PATCHES_PER_IMAGE_TRAIN: int = 60
    PATCHES_PER_IMAGE_VAL: int = 20
    PATCH_BATCH_SIZE: int = 32
    VESSEL_OVERSAMPLE_RATIO: float = 0.5

    IN_CHANNELS: int = 3
    ENCODER: str = "resnet34"
    ENCODER_WEIGHTS: str = "imagenet"
    DECODER_CHANNELS: Tuple[int, ...] = (256, 128, 64, 32, 16)

    # ---- model ablation flags (defaults = full model) ----
    USE_SKELETON_HEAD: bool = True
    USE_BTGI: bool = True
    USE_JAA: bool = True
    BTGI_STAGES: Tuple[int, ...] = (2, 3, 4)
    JAA_LAMBDA: float = 1.0

    # ---- loss weights ----
    LAMBDA_MASK: float = 1.0
    LAMBDA_SKELETON: float = 0.5
    LAMBDA_CONSISTENCY: float = 0.3
    LAMBDA_CLDICE: float = 0.5
    LAMBDA_ATCL: float = 1.0     # 0.0 disables ATCL, falls back to fixed-weight skeleton loss
    LAMBDA_JUNCTION: float = 0.1

    # ---- training ----
    EPOCHS: int = 30
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 1e-5
    OPTIMIZER: str = "adamw"
    SCHEDULER: str = "cosine"
    ETA_MIN: float = 1e-6
    USE_AMP: bool = True
    EARLY_STOPPING_PATIENCE: int = 8

    MIN_VESSEL_SIZE: int = 20  # skeleton morphology cleanup threshold
    JUNCTION_DEGREE_THRESHOLD: int = 3

    def create_dirs(self):
        for d in [self.OUTPUT_DIR, self.CHECKPOINT_DIR, self.LOG_DIR,
                  self.PRECOMPUTED_DIR, self.FIGURES_DIR]:
            os.makedirs(d, exist_ok=True)

    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_')}


cfg = CFG()
cfg.create_dirs()
seed_everything(cfg.SEED)

print("=" * 60)
print("CONFIGURATION (TopoEyeNet v2, unified patch pipeline)")
print("=" * 60)
for k, v in cfg.to_dict().items():
    print(f"  {k}: {v}")
print("=" * 60)
print("\nCELL 2 COMPLETE")


CONFIGURATION (TopoEyeNet v2, unified patch pipeline)
  EXPERIMENT_NAME: topoeyenet_v2
  SEED: 42
  SEEDS: (42, 43, 44)
  RETINOMIX_IMAGE_DIR: /kaggle/input/datasets/parikshakaur/retinomix-5/RETINOMIX-5/images
  RETINOMIX_MASK_DIR: /kaggle/input/datasets/parikshakaur/retinomix-5/RETINOMIX-5/mask
  DRIVE_TRAIN_IMAGE_DIR: /kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/training/images
  DRIVE_TRAIN_MASK_DIR: /kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/training/1st_manual
  DRIVE_TEST_IMAGE_DIR: /kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/test/images
  DRIVE_TEST_MASK_DIR: /kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/test/mask
  OUTPUT_DIR: /kaggle/working/output
  CHECKPOINT_DIR: /kaggle/working/checkpoints
  LOG_DIR: /kaggle/working/logs
  PRECOMPUTED_DIR: /kaggle/working/precomputed
  FIGURES_DIR: /kaggle/work

In [3]:
# ============================================================
# CELL 3: DATASET DISCOVERY & TRAIN/VAL/TEST SPLIT
# ============================================================
# NEEDS REAL DATA to run past the "discover_images" calls -- paths are
# the Kaggle input paths already used by the original notebook.

def discover_images(directory, extensions=('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp', '.gif')):
    if not os.path.exists(directory):
        return []
    files = []
    for root, _, filenames in os.walk(directory):
        for f in filenames:
            if f.lower().endswith(extensions):
                files.append(os.path.join(root, f))
    return sorted(files)


def extract_id(path):
    """Source- and split-aware ID extraction so different datasets/splits
    never collide on a shared numeric filename."""
    basename = os.path.splitext(os.path.basename(path))[0].lower()
    leading_match = re.match(r'^(\d+)', basename)
    if leading_match:
        return f"leading_{leading_match.group(1)}"
    key = re.sub(r'(image|mask)', '', basename)
    key = re.sub(r'_+', '_', key).strip('_')
    return key


def match_images_to_masks(image_paths, mask_paths):
    mask_lookup = {extract_id(mp): mp for mp in mask_paths}
    pairs, unmatched = [], []
    for ip in image_paths:
        iid = extract_id(ip)
        if iid in mask_lookup:
            pairs.append((ip, mask_lookup[iid]))
        else:
            unmatched.append(ip)
    return pairs, unmatched


def build_dataset_pairs(name, image_dir, mask_dir):
    image_paths = discover_images(image_dir)
    mask_paths = discover_images(mask_dir) if mask_dir else []
    print(f"{name}: {len(image_paths)} images, {len(mask_paths)} masks")
    if not image_paths or not mask_paths:
        print(f"  WARNING: {name} not found at given path(s) -- skipping. "
              f"(expected if you're not running this on Kaggle with the dataset attached)")
        return []
    pairs, unmatched = match_images_to_masks(image_paths, mask_paths)
    print(f"  Matched pairs: {len(pairs)} | Unmatched: {len(unmatched)}")
    return pairs


print("=" * 60)
print("DATASET DISCOVERY")
print("=" * 60)

retinomix_pairs = build_dataset_pairs("RetinoMix-5", cfg.RETINOMIX_IMAGE_DIR, cfg.RETINOMIX_MASK_DIR)
drive_train_pairs = build_dataset_pairs("DRIVE (train)", cfg.DRIVE_TRAIN_IMAGE_DIR, cfg.DRIVE_TRAIN_MASK_DIR)
drive_test_pairs = build_dataset_pairs("DRIVE (test, external validation)", cfg.DRIVE_TEST_IMAGE_DIR, cfg.DRIVE_TEST_MASK_DIR)

# ---- primary train/val/test split (RetinoMix-5): 70/15/15, seeded ----
from sklearn.model_selection import train_test_split

if retinomix_pairs:
    rm_trainval, rm_test = train_test_split(retinomix_pairs, test_size=0.15, random_state=cfg.SEED)
    rm_train, rm_val = train_test_split(rm_trainval, test_size=0.1765, random_state=cfg.SEED)  # 0.15/0.85
    rm_train_images, rm_train_masks = zip(*rm_train) if rm_train else ([], [])
    rm_val_images, rm_val_masks = zip(*rm_val) if rm_val else ([], [])
    rm_test_images, rm_test_masks = zip(*rm_test) if rm_test else ([], [])
    print(f"\nRetinoMix-5 split: train={len(rm_train)}  val={len(rm_val)}  test={len(rm_test)}")
else:
    rm_train_images, rm_train_masks = [], []
    rm_val_images, rm_val_masks = [], []
    rm_test_images, rm_test_masks = [], []
    print("\nNo RetinoMix-5 data found -- populate CFG paths / attach the Kaggle dataset before training.")

# DRIVE test set doubles as the external/cross-dataset validation set
if drive_test_pairs:
    drive_test_images, drive_test_masks = zip(*drive_test_pairs)
else:
    drive_test_images, drive_test_masks = [], []

print("\nCELL 3 COMPLETE")


DATASET DISCOVERY
RetinoMix-5: 233 images, 233 masks
  Matched pairs: 233 | Unmatched: 0
DRIVE (train): 20 images, 20 masks
  Matched pairs: 20 | Unmatched: 0
DRIVE (test, external validation): 20 images, 20 masks
  Matched pairs: 20 | Unmatched: 0

RetinoMix-5 split: train=163  val=35  test=35

CELL 3 COMPLETE


In [4]:
# ============================================================
# CELL 4: TOPOLOGY REPRESENTATION PRECOMPUTATION
# ============================================================
# For every mask, precompute ONCE (cached to .npz, reused every epoch):
#   mask         - binary vessel mask (uint8)
#   skeleton     - 1px-wide medial skeleton (skimage.morphology.skeletonize)
#   radius_map   - local vessel radius (distance transform), feeds ATCL
#   degree_map   - skeleton neighbor-count map, feeds ATCL + JAA supervision
#   junction_map - binary bifurcation/crossing map (degree >= threshold)
#   fov          - field-of-view mask (excludes black border), from the
#                  green-channel threshold + largest-connected-component
#                  procedure already validated in the original notebook

def preprocess_fundus(image_rgb: np.ndarray) -> np.ndarray:
    """CLAHE-enhanced green channel, replicated to 3 channels."""
    green = image_rgb[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)
    return enhanced  # uint8, single channel -- caller replicates to 3ch as needed


def compute_fov_mask(image_rgb: np.ndarray, threshold: int = 15) -> np.ndarray:
    green = image_rgb[:, :, 1]
    _, binary = cv2.threshold(green, threshold, 255, cv2.THRESH_BINARY)
    binary = binary.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num_labels > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        fov = (labels == largest).astype(np.uint8)
    else:
        fov = binary
    kernel = np.ones((9, 9), np.uint8)
    fov = cv2.erode(fov, kernel, iterations=1)
    return fov


def compute_skeleton(mask: np.ndarray, min_size: int = 20) -> np.ndarray:
    mask_bool = mask.astype(bool)
    if mask_bool.sum() > 0:
        mask_bool = remove_small_objects(mask_bool, min_size=min_size)
    return skeletonize(mask_bool).astype(np.uint8)


def compute_radius_map(mask: np.ndarray) -> np.ndarray:
    """Local vessel radius = distance transform of the mask, defined
    everywhere inside the vessel (used to upweight thin vessels in ATCL)."""
    return distance_transform_edt(mask.astype(bool)).astype(np.float32)


def compute_degree_map(skel: np.ndarray) -> np.ndarray:
    kernel = np.ones((3, 3), dtype=np.uint8)
    kernel[1, 1] = 0
    neighbor_count = convolve(skel.astype(np.uint8), kernel, mode="constant", cval=0)
    return (neighbor_count * skel).astype(np.float32)


def compute_junction_map(degree_map: np.ndarray, threshold: int = 3) -> np.ndarray:
    return (degree_map >= threshold).astype(np.uint8)


def precompute_representations(image_paths, mask_paths, out_dir, min_vessel_size=20,
                                junction_threshold=3, overwrite=False):
    os.makedirs(out_dir, exist_ok=True)
    n_done, n_skipped, n_failed = 0, 0, 0
    t0 = time.time()
    for i, (ip, mp) in enumerate(zip(image_paths, mask_paths)):
        base = os.path.splitext(os.path.basename(mp))[0]
        out_path = os.path.join(out_dir, f"{base}_reps.npz")
        if os.path.exists(out_path) and not overwrite:
            n_skipped += 1
            continue
        try:
            mask = np.array(Image.open(mp).convert('L'))
            mask = (mask > 127).astype(np.uint8)
            skel = compute_skeleton(mask, min_size=min_vessel_size)
            radius = compute_radius_map(mask)
            degree = compute_degree_map(skel)
            junction = compute_junction_map(degree, threshold=junction_threshold)

            np.savez_compressed(
                out_path,
                mask=mask.astype(np.uint8),
                skeleton=skel.astype(np.uint8),
                radius_map=radius.astype(np.float32),
                degree_map=degree.astype(np.float32),
                junction_map=junction.astype(np.uint8),
            )
            n_done += 1
        except Exception as e:
            print(f"  FAILED on {mp}: {e}")
            n_failed += 1

        if (i + 1) % 25 == 0:
            elapsed = time.time() - t0
            print(f"  {i+1}/{len(image_paths)} processed ({elapsed:.1f}s elapsed)")

    print(f"Done: {n_done} computed, {n_skipped} cached, {n_failed} failed -> {out_dir}")


print("=" * 60)
print("PRECOMPUTING TOPOLOGY REPRESENTATIONS")
print("=" * 60)

RM_PRECOMPUTED_DIR = os.path.join(cfg.PRECOMPUTED_DIR, "retinomix5")
DRIVE_PRECOMPUTED_DIR = os.path.join(cfg.PRECOMPUTED_DIR, "drive_test")

if len(rm_train_images) > 0:
    print("\nRetinoMix-5 train:")
    precompute_representations(rm_train_images, rm_train_masks, RM_PRECOMPUTED_DIR,
                                min_vessel_size=cfg.MIN_VESSEL_SIZE,
                                junction_threshold=cfg.JUNCTION_DEGREE_THRESHOLD)
    print("RetinoMix-5 val:")
    precompute_representations(rm_val_images, rm_val_masks, RM_PRECOMPUTED_DIR,
                                min_vessel_size=cfg.MIN_VESSEL_SIZE,
                                junction_threshold=cfg.JUNCTION_DEGREE_THRESHOLD)
    print("RetinoMix-5 test:")
    precompute_representations(rm_test_images, rm_test_masks, RM_PRECOMPUTED_DIR,
                                min_vessel_size=cfg.MIN_VESSEL_SIZE,
                                junction_threshold=cfg.JUNCTION_DEGREE_THRESHOLD)
else:
    print("Skipping -- no RetinoMix-5 data discovered in Cell 3.")

if len(drive_test_images) > 0:
    print("\nDRIVE test (external validation):")
    precompute_representations(drive_test_images, drive_test_masks, DRIVE_PRECOMPUTED_DIR,
                                min_vessel_size=cfg.MIN_VESSEL_SIZE,
                                junction_threshold=cfg.JUNCTION_DEGREE_THRESHOLD)

print("\nCELL 4 COMPLETE")


PRECOMPUTING TOPOLOGY REPRESENTATIONS

RetinoMix-5 train:
  25/163 processed (14.4s elapsed)
  50/163 processed (38.0s elapsed)
  75/163 processed (45.1s elapsed)
  100/163 processed (67.1s elapsed)
  125/163 processed (79.2s elapsed)
  150/163 processed (89.6s elapsed)
Done: 163 computed, 0 cached, 0 failed -> /kaggle/working/precomputed/retinomix5
RetinoMix-5 val:
  25/35 processed (4.4s elapsed)
Done: 35 computed, 0 cached, 0 failed -> /kaggle/working/precomputed/retinomix5
RetinoMix-5 test:
  25/35 processed (12.9s elapsed)
Done: 35 computed, 0 cached, 0 failed -> /kaggle/working/precomputed/retinomix5

DRIVE test (external validation):
Done: 20 computed, 0 cached, 0 failed -> /kaggle/working/precomputed/drive_test

CELL 4 COMPLETE


## Mathematical Formulation

Let $E(\cdot)$ be the shared ResNet encoder producing multi-scale features
$\{F^0, F^1, \dots, F^5\}$. Two parallel decoders, $D^m$ (mask) and $D^s$
(skeleton), each consume these features via standard U-Net decoder blocks,
producing per-stage features $F_l^m, F_l^s$ for stage $l \in \{0,\dots,4\}$
(coarse to fine).

### 1. Bidirectional Topology-Guided Interaction (BTGI)

At stages $l \in \mathcal{S} = \{2,3,4\}$ (the three highest-resolution decoder
stages, where thin vessels and junctions are still spatially resolved), each
branch's features are updated by a **learned, content-dependent gate** over
the other branch — not a fixed-weight sum or concatenation:

$$
g_l^{m\to s} = \sigma\big(W_{m\to s}\,[F_l^m \,;\, F_l^s]\big), \qquad
g_l^{s\to m} = \sigma\big(W_{s\to m}\,[F_l^s \,;\, F_l^m]\big)
$$

$$
\hat{F}_l^{s} = F_l^s + g_l^{m\to s} \odot \phi_m(F_l^m), \qquad
\hat{F}_l^{m} = F_l^m + g_l^{s\to m} \odot \phi_s(F_l^s)
$$

where $[\cdot;\cdot]$ is channel concatenation, $\sigma$ is the sigmoid, $\odot$
is the Hadamard product, and $\phi_m,\phi_s$ are $1\times1$ projections. Each
$W$ is itself a $1\times1$ conv over $2C_l$ channels, so the added cost per
stage is $O(C_l^2)$ — negligible next to the $O(9C_l^2)$ of a decoder block's
$3\times3$ convs at the same stage.

Interpretation: mask predictions "repair" fragmented skeleton predictions
(gap pixels get evidence from the mask branch even where local skeleton
evidence is weak), and skeleton predictions sharpen mask boundaries at
thin structures (skeleton confidence gates how much topology signal
flows back into the mask branch) — this is what makes it *bidirectional*
mask ↔ skeleton repair rather than the one-directional mask → skeleton
supervision the original framework used.

### 2. Adaptive Topology Consistency Loss (ATCL)

Fixed loss coefficients treat every pixel identically regardless of local
anatomy. ATCL instead computes a **per-pixel weight** from geometry that is
precomputed once from the ground truth (no extra runtime cost during
training):

$$
w_i = \operatorname{clip}\!\left(\frac{\alpha}{r_i + \epsilon} + \beta\, d_i,\; w_{\max}\right), \qquad
\tilde{w}_i = \frac{w_i}{\frac{1}{N}\sum_j w_j}
$$

where $r_i$ is the local vessel radius at pixel $i$ (Euclidean distance
transform of the GT mask) and $d_i$ is the local skeleton degree (bifurcation
density; $d_i \ge 3$ at junctions). Thin vessels ($r_i \to 0$) and junctions
($d_i \ge 3$) are automatically upweighted; thick, unambiguous vessel trunks
are downweighted. The loss combines a weighted BCE and a weighted Dice-style
term:

$$
\mathcal{L}_{\text{ATCL}} = \frac{1}{2N}\sum_i \tilde{w}_i \,\mathrm{BCE}(\hat{y}_i, y_i)
\;+\; \frac{1}{2}\left(1 - \frac{2\sum_i \tilde{w}_i \hat{p}_i y_i + 1}{\sum_i \tilde{w}_i \hat{p}_i + \sum_i \tilde{w}_i y_i + 1}\right)
$$

### 3. Junction-Aware Attention (JAA)

Rather than a generic (non-anatomical) spatial-attention block, JAA predicts
a junction probability map directly from skeleton-branch features and applies
it multiplicatively to mask features:

$$
J_l = \psi(F_l^s), \qquad \hat{F}_l^m = F_l^m \odot (1 + \lambda\,\sigma(J_l))
$$

$\psi$ is a small $3{\times}3\!\to\!1{\times}1$ conv stack, weakly supervised
against the precomputed junction map $y^{\text{junc}} = \mathbb{1}[d \ge 3]$
via an auxiliary Dice loss with weight $\lambda_{\text{junction}}$. This ties
the attention prior to an explicit anatomical target instead of leaving it
unconstrained.

### Total loss

$$
\mathcal{L} = \lambda_m \mathcal{L}_{\text{mask}} + \lambda_s \mathcal{L}_{\text{skel}} +
\lambda_c \mathcal{L}_{\text{consistency}} + \lambda_{cl}\mathcal{L}_{\text{clDice}} +
\lambda_j \mathcal{L}_{\text{junction}}
$$

with $\mathcal{L}_{\text{skel}} = \mathcal{L}_{\text{ATCL}}$ when ATCL is enabled,
else the fixed-weight Dice+BCE used in the original framework — this is exactly
the toggle the ablation study switches.

### Complexity

For encoder+decoder cost $O(P)$ parameters (≈24.4M for ResNet-34/U-Net at
these settings), BTGI + JAA add $\approx 34\text{K}$ parameters
(**+0.14%**, measured directly on the instantiated model — see the assertion
in the architecture cell), because every added module is a $1\times1$ or
small $3\times3$ conv operating on already-computed decoder features, not a
new encoder pass. Inference-time FLOPs overhead is dominated by the two
decoder branches themselves (present since the *original* dual-head design),
not by BTGI/JAA.


In [5]:
# ============================================================
# CELL 6: TOPOEYENET V2 ARCHITECTURE (BTGI + JAA)
# ============================================================
# Verified: forward + backward pass correctness checked for all 4
# ablation configurations (baseline / +skeleton / +BTGI / +BTGI+JAA)
# in a CPU sandbox before this notebook was written -- see the
# assertion block at the end of this cell, which re-runs the same
# check here so a broken environment fails loudly and immediately.

import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.decoders.unet.decoder import UnetDecoder, CenterBlock


class BTGI(nn.Module):
    """Bidirectional Topology-Guided Interaction.
    Gated cross-branch fusion between mask and skeleton decoder features
    at a given decoder stage. Not simple concatenation: each branch gates
    how much of the *other* branch's signal it accepts, conditioned on
    both feature maps jointly.
    """
    def __init__(self, channels):
        super().__init__()
        self.gate_m2s = nn.Sequential(
            nn.Conv2d(2 * channels, channels, kernel_size=1),
            nn.Sigmoid(),
        )
        self.gate_s2m = nn.Sequential(
            nn.Conv2d(2 * channels, channels, kernel_size=1),
            nn.Sigmoid(),
        )
        self.proj_m = nn.Conv2d(channels, channels, kernel_size=1)
        self.proj_s = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, f_m, f_s):
        g_m2s = self.gate_m2s(torch.cat([f_m, f_s], dim=1))
        g_s2m = self.gate_s2m(torch.cat([f_s, f_m], dim=1))
        f_s_new = f_s + g_m2s * self.proj_m(f_m)
        f_m_new = f_m + g_s2m * self.proj_s(f_s)
        return f_m_new, f_s_new


class JunctionAwareAttention(nn.Module):
    """Predicts a junction/bifurcation probability map from skeleton-branch
    features and uses it to reweight mask-branch features. Trained with a
    weak auxiliary supervision signal (junction/degree map) so the prior is
    anatomically grounded rather than a free-floating attention map.
    """
    def __init__(self, channels, lam=1.0):
        super().__init__()
        self.junction_conv = nn.Sequential(
            nn.Conv2d(channels, channels // 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 2, 1, kernel_size=1),
        )
        self.lam = lam

    def forward(self, f_mask, f_skel):
        junction_logits = self.junction_conv(f_skel)
        attn = torch.sigmoid(junction_logits)
        f_mask_out = f_mask * (1.0 + self.lam * attn)
        return f_mask_out, junction_logits


class TopoEyeNetV2(nn.Module):
    """
    Redesigned dual-branch retinal vessel segmentation network.

    Ablation flags let this single class instantiate every stage of the
    progressive ablation study:
      use_skeleton_head=False              -> plain single-branch U-Net (baseline)
      use_skeleton_head=True                -> + independent skeleton head (old TopoEyeNet)
      use_btgi=True                         -> + Bidirectional Topology-Guided Interaction
      use_jaa=True                          -> + Junction-Aware Attention (needs use_btgi context)
    """
    def __init__(self,
                 encoder_name="resnet34",
                 encoder_weights=None,
                 in_channels=3,
                 decoder_channels=(256, 128, 64, 32, 16),
                 use_skeleton_head=True,
                 use_btgi=True,
                 use_jaa=True,
                 btgi_stages=(2, 3, 4),
                 jaa_lambda=1.0):
        super().__init__()
        self.use_skeleton_head = use_skeleton_head
        self.use_btgi = use_btgi and use_skeleton_head
        self.use_jaa = use_jaa and self.use_btgi
        self.btgi_stages = set(btgi_stages)

        self.encoder = smp.encoders.get_encoder(
            encoder_name, in_channels=in_channels, depth=5, weights=encoder_weights,
        )
        enc_ch = self.encoder.out_channels

        self.mask_decoder = UnetDecoder(
            encoder_channels=enc_ch, decoder_channels=decoder_channels,
            n_blocks=5, use_batchnorm=True, attention_type=None, center=False,
        )
        if self.use_skeleton_head:
            self.skeleton_decoder = UnetDecoder(
                encoder_channels=enc_ch, decoder_channels=decoder_channels,
                n_blocks=5, use_batchnorm=True, attention_type=None, center=False,
            )

        if self.use_btgi:
            self.btgi_modules = nn.ModuleDict({
                str(i): BTGI(decoder_channels[i]) for i in self.btgi_stages
            })
        if self.use_jaa:
            last_stage = max(self.btgi_stages)
            self.jaa = JunctionAwareAttention(decoder_channels[last_stage], lam=jaa_lambda)
            self.jaa_stage = last_stage

        out_ch = decoder_channels[-1]
        self.mask_head = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True), nn.Conv2d(out_ch, 1, 1),
        )
        if self.use_skeleton_head:
            self.skeleton_head = nn.Sequential(
                nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True), nn.Conv2d(out_ch, 1, 1),
            )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    @staticmethod
    def _decode_step_prep(features):
        feats = features[1:][::-1]
        return {"head": feats[0], "skips": feats[1:]}

    def forward(self, x):
        features = self.encoder(x)

        mask_prep = self._decode_step_prep(features)
        x_m = self.mask_decoder.center(mask_prep["head"])

        if self.use_skeleton_head:
            skel_prep = self._decode_step_prep(features)
            x_s = self.skeleton_decoder.center(skel_prep["head"])
        else:
            skel_prep = None
            x_s = None

        junction_logits = None
        n_blocks = len(self.mask_decoder.blocks)
        for i in range(n_blocks):
            skip_m = mask_prep["skips"][i] if i < len(mask_prep["skips"]) else None
            x_m = self.mask_decoder.blocks[i](x_m, skip_m)

            if self.use_skeleton_head:
                skip_s = skel_prep["skips"][i] if i < len(skel_prep["skips"]) else None
                x_s = self.skeleton_decoder.blocks[i](x_s, skip_s)

                if self.use_btgi and i in self.btgi_stages:
                    x_m, x_s = self.btgi_modules[str(i)](x_m, x_s)

                if self.use_jaa and i == self.jaa_stage:
                    x_m, junction_logits = self.jaa(x_m, x_s)

        outputs = {"mask": self.mask_head(x_m)}
        if self.use_skeleton_head:
            outputs["skeleton"] = self.skeleton_head(x_s)
        if junction_logits is not None:
            outputs["junction"] = junction_logits
        return outputs



# ---- smoke test: run on import so a broken env fails immediately ----
def _smoke_test():
    torch.manual_seed(0)
    x = torch.randn(2, 3, 64, 64)
    configs = {
        "baseline":        dict(use_skeleton_head=False, use_btgi=False, use_jaa=False),
        "+skeleton_head":  dict(use_skeleton_head=True,  use_btgi=False, use_jaa=False),
        "+BTGI":           dict(use_skeleton_head=True,  use_btgi=True,  use_jaa=False),
        "+BTGI+JAA (full)":dict(use_skeleton_head=True,  use_btgi=True,  use_jaa=True),
    }
    param_counts = {}
    for name, kw in configs.items():
        m = TopoEyeNetV2(encoder_weights=None, **kw)
        out = m(x)
        loss = sum(v.sum() for v in out.values())
        loss.backward()
        assert m.encoder.conv1.weight.grad is not None, f"{name}: no grad reached encoder"
        param_counts[name] = sum(p.numel() for p in m.parameters())
        print(f"  {name:20s} outputs={list(out.keys())}  params={param_counts[name]:,}")
    overhead = param_counts["+BTGI+JAA (full)"] - param_counts["+skeleton_head"]
    pct = 100 * overhead / param_counts["+skeleton_head"]
    print(f"\nBTGI+JAA overhead vs plain dual-head: {overhead:,} params ({pct:.2f}%)")
    return param_counts

print("Running architecture smoke test (all 4 ablation configs)...")
_param_counts = _smoke_test()
print("CELL 6 COMPLETE -- architecture verified\n")


Running architecture smoke test (all 4 ablation configs)...
  baseline             outputs=['mask']  params=24,438,593
  +skeleton_head       outputs=['mask', 'skeleton']  params=27,592,514
  +BTGI                outputs=['mask', 'skeleton']  params=27,625,218
  +BTGI+JAA (full)     outputs=['mask', 'skeleton', 'junction']  params=27,626,403

BTGI+JAA overhead vs plain dual-head: 33,889 params (0.12%)
CELL 6 COMPLETE -- architecture verified



In [6]:
# ============================================================
# CELL 7: LOSS FUNCTIONS (DiceBCE, clDice, ATCL, StructuralConsistency)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0, eps=1e-7):
        super().__init__()
        self.smooth, self.eps = smooth, eps

    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred = pred.reshape(pred.size(0), -1)
        target = target.reshape(target.size(0), -1)
        inter = (pred * target).sum(dim=1)
        union = pred.sum(dim=1) + target.sum(dim=1)
        dice = (2 * inter + self.smooth) / (union + self.smooth + self.eps)
        return 1.0 - dice.mean()


class ClDiceLoss(nn.Module):
    """Topology-aware clDice loss (Shit et al. 2021), unweighted variant."""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, mask_logits, skeleton_logits, mask_target, skeleton_target):
        mask_prob = torch.sigmoid(mask_logits)
        skel_prob = torch.sigmoid(skeleton_logits)
        tprec = (torch.sum(skel_prob * mask_target) + self.smooth) / (torch.sum(skel_prob) + self.smooth)
        tsens = (torch.sum(skeleton_target * mask_prob) + self.smooth) / (torch.sum(skeleton_target) + self.smooth)
        return 1.0 - 2.0 * (tprec * tsens) / (tprec + tsens + self.smooth)


class AdaptiveTopologyConsistencyLoss(nn.Module):
    """
    ATCL: per-pixel loss weight derived from local anatomical geometry
    instead of a single fixed scalar coefficient.

        w_i = alpha / (r_i + eps) + beta * d_i

    r_i : local vessel radius (distance-transform value at pixel i,
          precomputed once from the GT mask -- thin vessels get large
          weight)
    d_i : local branch/junction density (precomputed skeleton degree
          map, 0 for non-skeleton pixels, >=3 at bifurcations)

    Both r and d are precomputed CPU-side per image (see representation
    pipeline) and passed in as tensors -- this loss does no morphology
    itself, it only consumes precomputed geometry, so it adds negligible
    training-time cost.
    """
    def __init__(self, alpha=1.0, beta=2.0, eps=1e-2, max_weight=10.0):
        super().__init__()
        self.alpha, self.beta, self.eps, self.max_weight = alpha, beta, eps, max_weight
        self.bce = nn.BCEWithLogitsLoss(reduction="none")

    def forward(self, pred_logits, target, radius_map, degree_map):
        w = self.alpha / (radius_map + self.eps) + self.beta * degree_map
        w = torch.clamp(w, max=self.max_weight)
        w = w / (w.mean() + 1e-7)  # keep loss magnitude comparable across images/batches

        pixel_bce = self.bce(pred_logits, target)
        weighted = (w * pixel_bce).mean()

        pred_prob = torch.sigmoid(pred_logits)
        num = 2 * (w * pred_prob * target).sum() + 1.0
        den = (w * pred_prob).sum() + (w * target).sum() + 1.0
        weighted_dice_loss = 1.0 - num / den

        return 0.5 * weighted + 0.5 * weighted_dice_loss


class StructuralConsistencyLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, mask_pred, skeleton_pred):
        mask_prob = torch.sigmoid(mask_pred)
        skel_prob = torch.sigmoid(skeleton_pred)
        violation = F.relu(skel_prob - mask_prob) * skel_prob
        return violation.sum() / (skel_prob.sum() + self.eps)


class TopoEyeLossV2(nn.Module):
    """
    Ablation-aware combined loss. Each novel term activates only when the
    corresponding target/prediction is present, so the SAME loss class is
    used at every ablation stage (baseline through full model) -- the
    ablation table differs only in which model outputs + targets exist,
    not in which loss class is instantiated.
    """
    def __init__(self, lambda_mask=1.0, lambda_skeleton=0.5, lambda_consistency=0.3,
                 lambda_cldice=0.5, lambda_atcl=0.0, lambda_junction=0.1):
        super().__init__()
        self.lambda_mask = lambda_mask
        self.lambda_skeleton = lambda_skeleton
        self.lambda_consistency = lambda_consistency
        self.lambda_cldice = lambda_cldice
        self.lambda_atcl = lambda_atcl
        self.lambda_junction = lambda_junction

        self.dice_bce_mask = DiceLoss()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice_bce_skel = DiceLoss()
        self.consistency = StructuralConsistencyLoss()
        self.cldice = ClDiceLoss()
        self.atcl = AdaptiveTopologyConsistencyLoss()
        self.junction_loss = DiceLoss()

    def forward(self, predictions, targets):
        losses = {}
        total = 0.0

        if "mask" in predictions and "mask" in targets:
            d = self.dice_bce_mask(predictions["mask"], targets["mask"])
            b = self.bce(predictions["mask"], targets["mask"])
            losses["mask"] = 0.7 * d + 0.3 * b
            total = total + self.lambda_mask * losses["mask"]

        if "skeleton" in predictions and "skeleton" in targets:
            if self.lambda_atcl > 0 and "radius_map" in targets and "degree_map" in targets:
                losses["skeleton"] = self.atcl(
                    predictions["skeleton"], targets["skeleton"],
                    targets["radius_map"], targets["degree_map"],
                )
            else:
                d = self.dice_bce_skel(predictions["skeleton"], targets["skeleton"])
                b = self.bce(predictions["skeleton"], targets["skeleton"])
                losses["skeleton"] = 0.7 * d + 0.3 * b
            total = total + self.lambda_skeleton * losses["skeleton"]

        if "mask" in predictions and "skeleton" in predictions:
            losses["consistency"] = self.consistency(predictions["mask"], predictions["skeleton"])
            total = total + self.lambda_consistency * losses["consistency"]

            if self.lambda_cldice > 0 and "mask" in targets and "skeleton" in targets:
                losses["cldice"] = self.cldice(
                    predictions["mask"], predictions["skeleton"],
                    targets["mask"], targets["skeleton"],
                )
                total = total + self.lambda_cldice * losses["cldice"]

        if "junction" in predictions and "junction" in targets:
            losses["junction"] = self.junction_loss(predictions["junction"], targets["junction"])
            total = total + self.lambda_junction * losses["junction"]

        losses["total"] = total
        return losses



# ---- smoke test: exercise every ablation stage's loss path ----
def _smoke_test_losses():
    torch.manual_seed(0)
    B, H, W = 2, 32, 32
    preds = {
        "mask": torch.randn(B, 1, H, W, requires_grad=True),
        "skeleton": torch.randn(B, 1, H, W, requires_grad=True),
        "junction": torch.randn(B, 1, H, W, requires_grad=True),
    }
    targets = {
        "mask": (torch.rand(B, 1, H, W) > 0.7).float(),
        "skeleton": (torch.rand(B, 1, H, W) > 0.9).float(),
        "junction": (torch.rand(B, 1, H, W) > 0.95).float(),
        "radius_map": torch.rand(B, 1, H, W) * 5 + 0.5,
        "degree_map": (torch.rand(B, 1, H, W) > 0.95).float() * 3,
    }

    stages = [
        ("baseline", TopoEyeLossV2(), {"mask": preds["mask"]}, {"mask": targets["mask"]}),
        ("+skeleton+consistency+clDice", TopoEyeLossV2(lambda_atcl=0.0),
         {"mask": preds["mask"], "skeleton": preds["skeleton"]},
         {"mask": targets["mask"], "skeleton": targets["skeleton"]}),
        ("full (+ATCL+junction)", TopoEyeLossV2(lambda_atcl=1.0), preds, targets),
    ]
    for name, crit, out, tgt in stages:
        losses = crit(out, tgt)
        assert torch.isfinite(losses["total"]), f"{name}: non-finite loss"
        print(f"  {name:32s} " + ", ".join(f"{k}={v.item():.4f}" for k, v in losses.items()))

print("Running loss-function smoke test (all 3 ablation stages)...")
_smoke_test_losses()
print("CELL 7 COMPLETE -- losses verified\n")


Running loss-function smoke test (all 3 ablation stages)...
  baseline                         mask=0.6781, total=0.6781
  +skeleton+consistency+clDice     mask=0.6781, skeleton=0.8278, consistency=0.1533, cldice=0.8296, total=1.5528
  full (+ATCL+junction)            mask=0.6781, skeleton=0.8242, consistency=0.1533, cldice=0.8296, junction=0.9169, total=1.6427
CELL 7 COMPLETE -- losses verified



In [7]:
# ============================================================
# CELL 8: EVALUATION METRICS
# ============================================================
# Dice, IoU, Precision, Recall, Specificity, Accuracy, F1, clDice,
# Betti-0 (connectivity) error, connectivity score, Hausdorff-95, ASSD.
import numpy as np
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize
from skimage import measure


def _binarize(x, thr=0.5):
    return (x > thr).astype(np.uint8)


def confusion_counts(pred, target):
    pred, target = _binarize(pred), _binarize(target)
    tp = np.sum((pred == 1) & (target == 1))
    fp = np.sum((pred == 1) & (target == 0))
    fn = np.sum((pred == 0) & (target == 1))
    tn = np.sum((pred == 0) & (target == 0))
    return tp, fp, fn, tn


def dice_score(pred, target, eps=1e-7):
    tp, fp, fn, _ = confusion_counts(pred, target)
    return (2 * tp + eps) / (2 * tp + fp + fn + eps)


def iou_score(pred, target, eps=1e-7):
    tp, fp, fn, _ = confusion_counts(pred, target)
    return (tp + eps) / (tp + fp + fn + eps)


def precision_recall_specificity_accuracy(pred, target, eps=1e-7):
    tp, fp, fn, tn = confusion_counts(pred, target)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    specificity = (tn + eps) / (tn + fp + eps)
    accuracy = (tp + tn) / (tp + fp + fn + tn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    return dict(precision=precision, recall=recall, specificity=specificity,
                accuracy=accuracy, f1=f1)


def cl_dice_metric(pred_mask, target_mask, pred_skel=None, target_skel=None, eps=1e-7):
    """clDice (Shit et al., 2021): topology-preserving Dice via skeletons."""
    pred_mask, target_mask = _binarize(pred_mask), _binarize(target_mask)
    pred_skel = skeletonize(pred_mask).astype(np.uint8) if pred_skel is None else _binarize(pred_skel)
    target_skel = skeletonize(target_mask).astype(np.uint8) if target_skel is None else _binarize(target_skel)

    tprec = (np.sum(pred_skel * target_mask) + eps) / (np.sum(pred_skel) + eps)
    tsens = (np.sum(target_skel * pred_mask) + eps) / (np.sum(target_skel) + eps)
    return 2 * tprec * tsens / (tprec + tsens + eps)


def betti0_error(pred, target):
    """|#connected components(pred) - #connected components(target)|,
    a simple, cheap proxy for Betti-0 (connectivity) topology error."""
    pred, target = _binarize(pred), _binarize(target)
    n_pred = measure.label(pred, connectivity=2).max()
    n_target = measure.label(target, connectivity=2).max()
    return abs(int(n_pred) - int(n_target))


def connectivity_score(pred, target, eps=1e-7):
    """Fraction of target connected components that have >=1 overlapping
    predicted pixel -- how much of the GT vessel tree survived."""
    target_bin = _binarize(target)
    pred_bin = _binarize(pred)
    labeled = measure.label(target_bin, connectivity=2)
    n_components = labeled.max()
    if n_components == 0:
        return 1.0
    recovered = 0
    for i in range(1, n_components + 1):
        comp_mask = labeled == i
        if np.any(pred_bin[comp_mask]):
            recovered += 1
    return recovered / n_components


def hausdorff_and_assd(pred, target, spacing=1.0, max_points=20000):
    """Boundary-based Hausdorff distance (95th percentile) and average
    symmetric surface distance, computed via distance transforms
    (avoids O(n^2) point-set methods)."""
    pred, target = _binarize(pred), _binarize(target)
    if pred.sum() == 0 or target.sum() == 0:
        return dict(hausdorff95=np.nan, assd=np.nan)

    dt_target = distance_transform_edt(1 - target) * spacing
    dt_pred = distance_transform_edt(1 - pred) * spacing

    pred_to_target = dt_target[pred > 0]
    target_to_pred = dt_pred[target > 0]

    all_d = np.concatenate([pred_to_target, target_to_pred])
    hausdorff95 = np.percentile(all_d, 95)
    assd = all_d.mean()
    return dict(hausdorff95=float(hausdorff95), assd=float(assd))


def compute_all_metrics(pred_mask, target_mask, pred_skel=None, target_skel=None):
    out = {}
    out["dice"] = dice_score(pred_mask, target_mask)
    out["iou"] = iou_score(pred_mask, target_mask)
    out.update(precision_recall_specificity_accuracy(pred_mask, target_mask))
    out["clDice"] = cl_dice_metric(pred_mask, target_mask, pred_skel, target_skel)
    out["betti0_error"] = betti0_error(pred_mask, target_mask)
    out["connectivity"] = connectivity_score(pred_mask, target_mask)
    hd = hausdorff_and_assd(pred_mask, target_mask)
    out.update(hd)
    return out



# ---- smoke test: demonstrates WHY topology metrics matter, using a
# synthetic "broken vessel" case where Dice barely moves but the
# topology metrics correctly flag the break ----
def _smoke_test_metrics():
    H, W = 128, 128
    target = np.zeros((H, W), dtype=np.uint8)
    target[40:90, 60:64] = 1
    target[60:64, 20:100] = 1
    pred_good = target.copy()
    pred_broken = target.copy()
    pred_broken[70:75, 60:64] = 0  # simulate a broken vessel segment

    m_good = compute_all_metrics(pred_good.astype(float), target.astype(float))
    m_broken = compute_all_metrics(pred_broken.astype(float), target.astype(float))

    print(f"  {'metric':14s} {'intact':>10s} {'broken':>10s}")
    for k in m_good:
        print(f"  {k:14s} {m_good[k]:>10.4f} {m_broken[k]:>10.4f}")

    dice_drop = m_good["dice"] - m_broken["dice"]
    assert m_broken["betti0_error"] >= 1, "broken vessel should increase component count"
    print(f"\n  Dice dropped only {dice_drop:.4f} for a fully-severed vessel, while "
          f"betti0_error correctly went from {m_good['betti0_error']} to "
          f"{m_broken['betti0_error']}. This is the core motivation for reporting "
          f"topology metrics alongside Dice in every table below.")

print("Running metrics smoke test (broken-vessel demonstration)...")
_smoke_test_metrics()
print("CELL 8 COMPLETE -- metrics verified\n")


Running metrics smoke test (broken-vessel demonstration)...
  metric             intact     broken
  dice               1.0000     0.9798
  iou                1.0000     0.9603
  precision          1.0000     1.0000
  recall             1.0000     0.9603
  specificity        1.0000     1.0000
  accuracy           1.0000     0.9988
  f1                 1.0000     0.9798
  clDice             1.0000     0.9796
  betti0_error       0.0000     1.0000
  connectivity       1.0000     1.0000
  hausdorff95        0.0000     0.0000
  assd               0.0000     0.0364

  Dice dropped only 0.0202 for a fully-severed vessel, while betti0_error correctly went from 0 to 1. This is the core motivation for reporting topology metrics alongside Dice in every table below.
CELL 8 COMPLETE -- metrics verified



In [8]:
# ============================================================
# CELL 9: UNIFIED PATCH DATASET
# ============================================================
# Single dataset class for every ablation stage. Returns whichever
# targets exist in the precomputed .npz; TopoEyeLossV2 and the model's
# ablation flags decide what's actually used, so this class never needs
# to change between baseline and full-model runs.

from torch.utils.data import Dataset, DataLoader


class TopoPatchDataset(Dataset):
    def __init__(self, image_paths, mask_paths, precomputed_dir,
                 patch_size=128, patches_per_image=60, vessel_oversample_ratio=0.5,
                 augment=True, warmup=True):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.precomputed_dir = precomputed_dir
        self.patch_size = patch_size
        self.patches_per_image = patches_per_image
        self.vessel_oversample_ratio = vessel_oversample_ratio
        self.augment = augment
        self._cache = {}

        self.valid_indices = []
        for i, mp in enumerate(mask_paths):
            if mp is None:
                continue
            base = os.path.splitext(os.path.basename(mp))[0]
            npz_path = os.path.join(precomputed_dir, f"{base}_reps.npz")
            if os.path.exists(npz_path):
                self.valid_indices.append(i)

        print(f"TopoPatchDataset: {len(self.valid_indices)} source images, "
              f"{patches_per_image} patches/image = "
              f"{len(self.valid_indices) * patches_per_image} patches/epoch")

        if warmup and self.valid_indices:
            self._warmup_cache()

    def _warmup_cache(self):
        t0 = time.time()
        for n, real_idx in enumerate(self.valid_indices):
            self._load_all(real_idx)
            if (n + 1) % 20 == 0 or (n + 1) == len(self.valid_indices):
                print(f"    cached {n+1}/{len(self.valid_indices)} "
                      f"({time.time()-t0:.1f}s elapsed)")

    def __len__(self):
        return max(1, len(self.valid_indices)) * self.patches_per_image

    def _load_all(self, real_idx):
        if real_idx in self._cache:
            return self._cache[real_idx]

        img_path = self.image_paths[real_idx]
        mask_path = self.mask_paths[real_idx]

        image = np.array(Image.open(img_path).convert('RGB'))
        base = os.path.splitext(os.path.basename(mask_path))[0]
        npz_path = os.path.join(self.precomputed_dir, f"{base}_reps.npz")
        reps = dict(np.load(npz_path))

        h, w = image.shape[:2]

        def _resize_if_needed(arr, interp):
            if arr.shape[0] != h or arr.shape[1] != w:
                return cv2.resize(arr, (w, h), interpolation=interp)
            return arr

        mask = _resize_if_needed(reps['mask'].astype(np.uint8), cv2.INTER_NEAREST)
        skeleton = _resize_if_needed(reps['skeleton'].astype(np.uint8), cv2.INTER_NEAREST)
        radius_map = _resize_if_needed(reps['radius_map'].astype(np.float32), cv2.INTER_LINEAR)
        degree_map = _resize_if_needed(reps['degree_map'].astype(np.float32), cv2.INTER_NEAREST)
        junction_map = _resize_if_needed(reps['junction_map'].astype(np.uint8), cv2.INTER_NEAREST)

        green = image[:, :, 1]
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        preprocessed_1ch = clahe.apply(green)
        fov = compute_fov_mask(image)

        result = dict(img=preprocessed_1ch, mask=mask, skeleton=skeleton,
                      radius_map=radius_map, degree_map=degree_map,
                      junction_map=junction_map, fov=fov)
        self._cache[real_idx] = result
        return result

    def _augment_patch(self, patches):
        if random.random() < 0.5:
            patches = [np.fliplr(p).copy() for p in patches]
        if random.random() < 0.5:
            patches = [np.flipud(p).copy() for p in patches]
        k = random.randint(0, 3)
        if k > 0:
            patches = [np.rot90(p, k).copy() for p in patches]
        return patches

    def __getitem__(self, idx):
        if not self.valid_indices:
            raise RuntimeError("No valid samples -- did Cell 4 (precompute) run against real data?")
        real_idx = self.valid_indices[idx % len(self.valid_indices)]
        data = self._load_all(real_idx)
        mask, fov = data['mask'], data['fov']
        h, w = mask.shape
        ps = self.patch_size

        vessel_ys, vessel_xs = np.where((mask > 0) & (fov > 0))
        use_vessel_center = len(vessel_ys) > 0 and random.random() < self.vessel_oversample_ratio

        y0, x0 = 0, 0
        for _ in range(20):
            if use_vessel_center:
                i = random.randint(0, len(vessel_ys) - 1)
                cy, cx = vessel_ys[i], vessel_xs[i]
            else:
                cy = random.randint(ps // 2, max(ps // 2, h - ps // 2 - 1))
                cx = random.randint(ps // 2, max(ps // 2, w - ps // 2 - 1))
            y0 = max(0, min(h - ps, cy - ps // 2))
            x0 = max(0, min(w - ps, cx - ps // 2))
            if fov[y0:y0 + ps, x0:x0 + ps].mean() > 0.5:
                break

        def crop(key, dtype):
            return data[key][y0:y0 + ps, x0:x0 + ps].astype(dtype).copy()

        img_1ch = crop('img', np.uint8)
        mask_p = crop('mask', np.float32)
        skel_p = crop('skeleton', np.float32)
        radius_p = crop('radius_map', np.float32)
        degree_p = crop('degree_map', np.float32)
        junction_p = crop('junction_map', np.float32)

        if self.augment:
            img_1ch, mask_p, skel_p, radius_p, degree_p, junction_p = self._augment_patch(
                [img_1ch, mask_p, skel_p, radius_p, degree_p, junction_p]
            )

        img_patch = np.stack([img_1ch] * 3, axis=-1).astype(np.float32) / 255.0
        img_tensor = torch.from_numpy(img_patch.transpose(2, 0, 1)).float()

        targets = {
            'mask': torch.from_numpy(mask_p).unsqueeze(0).float(),
            'skeleton': torch.from_numpy(skel_p).unsqueeze(0).float(),
            'radius_map': torch.from_numpy(radius_p).unsqueeze(0).float(),
            'degree_map': torch.from_numpy(degree_p).unsqueeze(0).float(),
            'junction': torch.from_numpy(junction_p).unsqueeze(0).float(),
        }
        return img_tensor, targets


def build_patch_loaders(train_images, train_masks, val_images, val_masks, precomputed_dir, cfg):
    train_ds = TopoPatchDataset(
        train_images, train_masks, precomputed_dir,
        patch_size=cfg.PATCH_SIZE, patches_per_image=cfg.PATCHES_PER_IMAGE_TRAIN,
        vessel_oversample_ratio=cfg.VESSEL_OVERSAMPLE_RATIO, augment=True, warmup=True,
    )
    val_ds = TopoPatchDataset(
        val_images, val_masks, precomputed_dir,
        patch_size=cfg.PATCH_SIZE, patches_per_image=cfg.PATCHES_PER_IMAGE_VAL,
        vessel_oversample_ratio=cfg.VESSEL_OVERSAMPLE_RATIO, augment=False, warmup=True,
    )
    train_loader = DataLoader(train_ds, batch_size=cfg.PATCH_BATCH_SIZE, shuffle=True,
                               num_workers=0, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.PATCH_BATCH_SIZE, shuffle=False,
                             num_workers=0, pin_memory=True)
    return train_loader, val_loader


# ---- smoke test with synthetic data (no real dataset required) ----
def _smoke_test_patch_dataset():
    tmp_dir = "/tmp/topoeye_smoke"
    os.makedirs(tmp_dir, exist_ok=True)
    img_dir = os.path.join(tmp_dir, "images")
    os.makedirs(img_dir, exist_ok=True)
    rep_dir = os.path.join(tmp_dir, "reps")
    os.makedirs(rep_dir, exist_ok=True)

    H, W = 256, 256
    rng = np.random.default_rng(0)
    image = (rng.random((H, W, 3)) * 200 + 30).astype(np.uint8)
    mask = np.zeros((H, W), dtype=np.uint8)
    mask[80:180, 120:126] = 1
    mask[120:126, 40:220] = 1

    img_path = os.path.join(img_dir, "sample_image.png")
    Image.fromarray(image).save(img_path)

    skel = compute_skeleton(mask, min_size=5)
    radius = compute_radius_map(mask)
    degree = compute_degree_map(skel)
    junction = compute_junction_map(degree, threshold=3)
    npz_path = os.path.join(rep_dir, "sample_mask_reps.npz")
    np.savez_compressed(npz_path, mask=mask, skeleton=skel, radius_map=radius,
                         degree_map=degree, junction_map=junction)
    mask_path = os.path.join(tmp_dir, "sample_mask.png")  # basename must match npz stem

    ds = TopoPatchDataset([img_path], [os.path.join(rep_dir, "sample_mask.png")],
                           rep_dir, patch_size=64, patches_per_image=5,
                           vessel_oversample_ratio=0.8, augment=True, warmup=True)
    img_t, tgt = ds[0]
    assert img_t.shape == (3, 64, 64), img_t.shape
    for k, v in tgt.items():
        assert v.shape == (1, 64, 64), f"{k}: {v.shape}"
    print(f"  patch dataset OK: img {img_t.shape}, targets {list(tgt.keys())}")
    print(f"  mask patch has {tgt['mask'].sum().item():.0f} vessel px "
          f"(oversampling toward vessel centers working)")


print("Running patch-dataset smoke test (synthetic data)...")
_smoke_test_patch_dataset()
print("CELL 9 COMPLETE -- patch pipeline verified\n")


Running patch-dataset smoke test (synthetic data)...
TopoPatchDataset: 1 source images, 5 patches/image = 5 patches/epoch
    cached 1/1 (0.0s elapsed)
  patch dataset OK: img torch.Size([3, 64, 64]), targets ['mask', 'skeleton', 'radius_map', 'degree_map', 'junction']
  mask patch has 240 vessel px (oversampling toward vessel centers working)
CELL 9 COMPLETE -- patch pipeline verified



In [9]:
# ============================================================
# CELL 10: TRAINING ENGINE
# ============================================================
# One function trains ANY ablation configuration (baseline through full
# model) -- the config dict controls model flags and loss weights, so
# the ablation study in Cell 12 just calls this in a loop.

from torch.cuda.amp import autocast, GradScaler


def dice_from_logits(logits, target, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    tp = (preds * target).sum()
    fp = (preds * (1 - target)).sum()
    fn = ((1 - preds) * target).sum()
    return (2 * tp / (2 * tp + fp + fn + eps)).item()


def train_one_config(stage_name, model_kwargs, loss_kwargs, train_loader, val_loader,
                      cfg, seed=42, verbose=True):
    """Trains one ablation stage. Returns (best_val_dice, history_df, ckpt_path)."""
    seed_everything(seed)

    model = TopoEyeNetV2(
        encoder_name=cfg.ENCODER, encoder_weights=cfg.ENCODER_WEIGHTS,
        in_channels=cfg.IN_CHANNELS, decoder_channels=cfg.DECODER_CHANNELS,
        btgi_stages=cfg.BTGI_STAGES, jaa_lambda=cfg.JAA_LAMBDA,
        **model_kwargs,
    )
    model = wrap_dual_gpu(model)

    criterion = TopoEyeLossV2(**loss_kwargs)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE,
                                   weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.EPOCHS, eta_min=cfg.ETA_MIN)
    scaler = GradScaler(enabled=cfg.USE_AMP)

    ckpt_path = os.path.join(cfg.CHECKPOINT_DIR, f"{stage_name}_seed{seed}.pt")
    # -1.0, not 0.0: a real val_dice of exactly 0.0 (easily hit in early/short
    # training) must still trigger a save on epoch 1, or no checkpoint file
    # ever gets written and the caller's torch.load() crashes downstream.
    best_val_dice = -1.0
    patience_counter = 0
    history = []

    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        t0 = time.time()
        train_losses = defaultdict(float)
        n_batches = 0

        for images, targets in train_loader:
            images = images.to(device, non_blocking=True)
            targets = {k: v.to(device, non_blocking=True) for k, v in targets.items()}

            optimizer.zero_grad()
            with autocast(enabled=cfg.USE_AMP):
                preds = model(images)
                losses = criterion(preds, targets)

            scaler.scale(losses['total']).backward()
            scaler.step(optimizer)
            scaler.update()

            for k, v in losses.items():
                train_losses[k] += v.item()
            n_batches += 1

        scheduler.step()
        for k in train_losses:
            train_losses[k] /= max(1, n_batches)

        model.eval()
        val_dice, n_val = 0.0, 0
        with torch.no_grad():
            for images, targets in val_loader:
                images = images.to(device, non_blocking=True)
                mask_t = targets['mask'].to(device, non_blocking=True)
                with autocast(enabled=cfg.USE_AMP):
                    preds = model(images)
                val_dice += dice_from_logits(preds['mask'], mask_t)
                n_val += 1
        val_dice /= max(1, n_val)

        history.append(dict(epoch=epoch, val_dice=val_dice,
                             time_s=time.time() - t0, **train_losses))

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            patience_counter = 0
            state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            torch.save({'state_dict': state, 'model_kwargs': model_kwargs,
                        'val_dice': val_dice, 'epoch': epoch}, ckpt_path)
        else:
            patience_counter += 1

        if verbose and (epoch == 1 or epoch % 5 == 0 or epoch == cfg.EPOCHS):
            print(f"  [{stage_name} seed={seed}] epoch {epoch}/{cfg.EPOCHS} "
                  f"| total={train_losses['total']:.4f} | val_dice={val_dice:.4f} "
                  f"(best={best_val_dice:.4f}) | {time.time()-t0:.1f}s")

        if patience_counter >= cfg.EARLY_STOPPING_PATIENCE:
            print(f"  Early stopping at epoch {epoch} (no improvement for "
                  f"{cfg.EARLY_STOPPING_PATIENCE} epochs)")
            break

        clear_gpu_cache()

    return best_val_dice, pd.DataFrame(history), ckpt_path


print("CELL 10 COMPLETE -- train_one_config() ready "
      "(runs any ablation stage: pass model_kwargs / loss_kwargs)")


CELL 10 COMPLETE -- train_one_config() ready (runs any ablation stage: pass model_kwargs / loss_kwargs)


In [10]:
# ============================================================
# CELL 11: OVERLAP-TILE INFERENCE + FULL-IMAGE EVALUATION
# ============================================================
# The model is trained on small native-resolution patches; at test time
# we tile the full image with overlap and average predictions in the
# overlap regions, which avoids the seam artifacts a naive non-overlapping
# tiling would produce along patch borders.

def overlap_tile_predict(model, image_1ch_uint8, patch_size=128, stride=96, batch_size=32):
    """image_1ch_uint8: single-channel CLAHE-preprocessed image (H, W) uint8.
    Returns dict of (H, W) float32 probability maps for every model output."""
    model.eval()
    h, w = image_1ch_uint8.shape
    pad_h = max(0, patch_size - h) + ((stride - (h - patch_size) % stride) % stride if h > patch_size else 0)
    pad_w = max(0, patch_size - w) + ((stride - (w - patch_size) % stride) % stride if w > patch_size else 0)
    padded = np.pad(image_1ch_uint8, ((0, pad_h), (0, pad_w)), mode='reflect')
    ph, pw = padded.shape

    coords = []
    for y in range(0, ph - patch_size + 1, stride):
        for x in range(0, pw - patch_size + 1, stride):
            coords.append((y, x))

    accum = {}
    weight = np.zeros((ph, pw), dtype=np.float32)

    with torch.no_grad():
        for i in range(0, len(coords), batch_size):
            batch_coords = coords[i:i + batch_size]
            batch_patches = []
            for y, x in batch_coords:
                patch = padded[y:y + patch_size, x:x + patch_size]
                patch3 = np.stack([patch] * 3, axis=-1).astype(np.float32) / 255.0
                batch_patches.append(patch3.transpose(2, 0, 1))
            batch_t = torch.from_numpy(np.stack(batch_patches)).float().to(device)

            with autocast(enabled=cfg.USE_AMP):
                out = model(batch_t)

            for key, logits in out.items():
                probs = torch.sigmoid(logits).cpu().numpy()[:, 0]  # (B, ps, ps)
                if key not in accum:
                    accum[key] = np.zeros((ph, pw), dtype=np.float32)
                for (y, x), p in zip(batch_coords, probs):
                    accum[key][y:y + patch_size, x:x + patch_size] += p

            for y, x in batch_coords:
                weight[y:y + patch_size, x:x + patch_size] += 1.0

    results = {}
    for key, acc in accum.items():
        avg = acc / np.maximum(weight, 1e-7)
        results[key] = avg[:h, :w]
    return results


def _align_to_shape(arr, target_hw, is_binary=True):
    """Resizes a precomputed representation (mask/skeleton/etc, at whatever
    resolution the mask FILE happened to be saved at) onto the shape of the
    image actually used for inference. Some datasets (e.g. RetinoMix-5) do
    not guarantee identical image/mask pixel dimensions per pair, so this
    alignment is not optional -- skipping it is what caused:
        ValueError: operands could not be broadcast together with shapes
        (443,512) (605,700)
    Uses the same INTER_NEAREST-for-binary convention as the training
    dataset's _resize_if_needed, so eval-time and train-time preprocessing
    stay consistent."""
    if arr.shape[:2] == tuple(target_hw):
        return arr
    interp = cv2.INTER_NEAREST if is_binary else cv2.INTER_LINEAR
    resized = cv2.resize(arr.astype(np.float32), (target_hw[1], target_hw[0]), interpolation=interp)
    return resized.astype(arr.dtype) if not is_binary else (resized > 0.5).astype(arr.dtype)


def evaluate_model_on_set(model, image_paths, mask_paths, precomputed_dir,
                           patch_size=128, stride=96, threshold=0.5):
    """Runs overlap-tile inference + full metric suite on every image,
    returns a per-image DataFrame (rows = images, cols = metrics)."""
    rows = []
    for ip, mp in zip(image_paths, mask_paths):
        base = os.path.splitext(os.path.basename(mp))[0]
        npz_path = os.path.join(precomputed_dir, f"{base}_reps.npz")
        if not os.path.exists(npz_path):
            continue
        reps = dict(np.load(npz_path))

        image = np.array(Image.open(ip).convert('RGB'))
        green = image[:, :, 1]
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        preprocessed = clahe.apply(green)
        target_hw = preprocessed.shape[:2]

        # Align GT representations to the image's actual shape -- NOT the
        # mask file's native shape, which may differ (see docstring above).
        gt_mask = _align_to_shape(reps['mask'].astype(np.float32), target_hw, is_binary=True)
        gt_skel = _align_to_shape(reps['skeleton'].astype(np.float32), target_hw, is_binary=True) \
            if 'skeleton' in reps else None

        pred = overlap_tile_predict(model, preprocessed, patch_size=patch_size, stride=stride)
        pred_mask = (pred['mask'] > threshold).astype(np.float32)
        pred_skel = (pred['skeleton'] > threshold).astype(np.float32) if 'skeleton' in pred else None

        assert pred_mask.shape == gt_mask.shape, \
            f"{os.path.basename(ip)}: pred {pred_mask.shape} vs gt {gt_mask.shape} after alignment -- should never happen"

        m = compute_all_metrics(pred_mask, gt_mask,
                                 pred_skel=pred_skel, target_skel=gt_skel)
        m['image'] = os.path.basename(ip)
        rows.append(m)

    return pd.DataFrame(rows)


print("CELL 11 COMPLETE -- overlap_tile_predict() / evaluate_model_on_set() ready "
      "(now shape-aligns GT to image dimensions before scoring)")


CELL 11 COMPLETE -- overlap_tile_predict() / evaluate_model_on_set() ready (now shape-aligns GT to image dimensions before scoring)


In [11]:
# ============================================================
# CELL 12: PROGRESSIVE ABLATION STUDY (multi-seed)
# ============================================================
# NEEDS REAL DATA + GPU to actually train. Structure and bookkeeping are
# fully wired; run this on Kaggle once Cells 3-4 have real datasets.

ABLATION_STAGES = {
    "A_baseline_mask_only": dict(
        model_kwargs=dict(use_skeleton_head=False, use_btgi=False, use_jaa=False),
        loss_kwargs=dict(lambda_skeleton=0.0, lambda_consistency=0.0,
                          lambda_cldice=0.0, lambda_atcl=0.0, lambda_junction=0.0),
    ),
    "B_plus_skeleton_head": dict(
        model_kwargs=dict(use_skeleton_head=True, use_btgi=False, use_jaa=False),
        loss_kwargs=dict(lambda_cldice=0.0, lambda_atcl=0.0, lambda_junction=0.0),
    ),
    "C_plus_cldice_consistency": dict(
        model_kwargs=dict(use_skeleton_head=True, use_btgi=False, use_jaa=False),
        loss_kwargs=dict(lambda_atcl=0.0, lambda_junction=0.0),
    ),
    "D_plus_BTGI": dict(
        model_kwargs=dict(use_skeleton_head=True, use_btgi=True, use_jaa=False),
        loss_kwargs=dict(lambda_atcl=0.0, lambda_junction=0.0),
    ),
    "E_plus_ATCL": dict(
        model_kwargs=dict(use_skeleton_head=True, use_btgi=True, use_jaa=False),
        loss_kwargs=dict(lambda_atcl=1.0, lambda_junction=0.0),
    ),
    "F_full_plus_JAA": dict(
        model_kwargs=dict(use_skeleton_head=True, use_btgi=True, use_jaa=True),
        loss_kwargs=dict(lambda_atcl=1.0, lambda_junction=0.1),
    ),
}


def run_ablation_study(cfg, train_loader, val_loader, test_images, test_masks,
                        precomputed_dir, seeds=None, stages=None):
    seeds = seeds or cfg.SEEDS
    stages = stages or ABLATION_STAGES

    all_results = []
    for stage_name, stage_cfg in stages.items():
        for seed in seeds:
            print(f"\n{'='*60}\nABLATION STAGE: {stage_name} | seed={seed}\n{'='*60}")
            best_val_dice, history, ckpt_path = train_one_config(
                stage_name, stage_cfg["model_kwargs"], stage_cfg["loss_kwargs"],
                train_loader, val_loader, cfg, seed=seed,
            )

            if not os.path.exists(ckpt_path):
                print(f"  WARNING: no checkpoint was saved for {stage_name} seed={seed} "
                      f"(training likely never improved past its starting point) -- "
                      f"skipping test-set evaluation for this run rather than crashing.")
                continue

            model = TopoEyeNetV2(
                encoder_name=cfg.ENCODER, encoder_weights=None,
                in_channels=cfg.IN_CHANNELS, decoder_channels=cfg.DECODER_CHANNELS,
                btgi_stages=cfg.BTGI_STAGES, jaa_lambda=cfg.JAA_LAMBDA,
                **stage_cfg["model_kwargs"],
            ).to(device)
            ckpt = torch.load(ckpt_path, map_location=device)
            model.load_state_dict(ckpt['state_dict'])

            test_df = evaluate_model_on_set(model, test_images, test_masks, precomputed_dir,
                                             patch_size=cfg.PATCH_SIZE, stride=int(cfg.PATCH_SIZE * 0.75))
            test_df['stage'] = stage_name
            test_df['seed'] = seed
            n_params = sum(p.numel() for p in model.parameters())
            test_df['n_params'] = n_params
            all_results.append(test_df)

            history.to_csv(os.path.join(cfg.LOG_DIR, f"history_{stage_name}_seed{seed}.csv"), index=False)
            del model
            clear_gpu_cache()

    if not all_results:
        raise RuntimeError(
            "No ablation stage produced a usable checkpoint -- every stage's "
            "val_dice apparently never improved. Check EPOCHS / dataset size / "
            "learning rate before re-running."
        )
    results_df = pd.concat(all_results, ignore_index=True)
    results_df.to_csv(os.path.join(cfg.LOG_DIR, "ablation_full_results.csv"), index=False)
    return results_df


def summarize_ablation(results_df, metrics=("dice", "iou", "clDice", "betti0_error", "connectivity")):
    summary = (results_df.groupby("stage")[list(metrics) + ["n_params"]]
               .agg(["mean", "std"]))
    stage_order = list(ABLATION_STAGES.keys())
    summary = summary.reindex([s for s in stage_order if s in summary.index])
    return summary


print("CELL 12 COMPLETE -- run_ablation_study(cfg, train_loader, val_loader, "
      "test_images, test_masks, precomputed_dir) ready.")
print("Example (once Cells 3-4 have real data):")
print("  results_df = run_ablation_study(cfg, patch_train_loader, patch_val_loader,")
print("                                   rm_test_images, rm_test_masks, RM_PRECOMPUTED_DIR)")
print("  print(summarize_ablation(results_df))")


CELL 12 COMPLETE -- run_ablation_study(cfg, train_loader, val_loader, test_images, test_masks, precomputed_dir) ready.
Example (once Cells 3-4 have real data):
  results_df = run_ablation_study(cfg, patch_train_loader, patch_val_loader,
                                   rm_test_images, rm_test_masks, RM_PRECOMPUTED_DIR)
  print(summarize_ablation(results_df))


In [12]:
# ============================================================
# CELL 13: STATISTICAL SIGNIFICANCE ANALYSIS
# ============================================================
from scipy import stats as scipy_stats


def normality_report(results_df, metric="dice", group_col="stage"):
    rows = []
    for stage, g in results_df.groupby(group_col):
        vals = g[metric].dropna().values
        if len(vals) < 3:
            rows.append(dict(stage=stage, n=len(vals), shapiro_p=np.nan, normal=None))
            continue
        stat, p = scipy_stats.shapiro(vals)
        rows.append(dict(stage=stage, n=len(vals), shapiro_p=p, normal=bool(p > 0.05)))
    return pd.DataFrame(rows)


def cohens_d_paired(a, b):
    diff = np.asarray(a) - np.asarray(b)
    return diff.mean() / (diff.std(ddof=1) + 1e-12)


def rank_biserial_from_wilcoxon(a, b):
    diff = np.asarray(a) - np.asarray(b)
    diff = diff[diff != 0]
    if len(diff) == 0:
        return 0.0
    n_pos = np.sum(diff > 0)
    n_neg = np.sum(diff < 0)
    return (n_pos - n_neg) / len(diff)


def effect_size_label(x):
    ax = abs(x)
    if ax < 0.2:
        return "negligible"
    if ax < 0.5:
        return "small"
    if ax < 0.8:
        return "medium"
    return "large"


def holm_bonferroni(pvalues):
    """Returns adjusted p-values (Holm-Bonferroni step-down), same order as input."""
    pvalues = np.asarray(pvalues)
    n = len(pvalues)
    order = np.argsort(pvalues)
    adjusted = np.empty(n)
    running_max = 0.0
    for rank, idx in enumerate(order):
        val = (n - rank) * pvalues[idx]
        running_max = max(running_max, val)
        adjusted[idx] = min(running_max, 1.0)
    return adjusted


def pairwise_stage_comparison(results_df, metric="dice", group_col="stage", stage_order=None):
    """Paired comparison between CONSECUTIVE ablation stages (same seeds),
    auto-selecting t-test vs Wilcoxon by normality of the paired differences,
    with Holm-Bonferroni correction across all pairs tested."""
    stages = stage_order or sorted(results_df[group_col].unique())
    rows = []
    for s1, s2 in zip(stages[:-1], stages[1:]):
        g1 = results_df[results_df[group_col] == s1].groupby("seed")[metric].mean()
        g2 = results_df[results_df[group_col] == s2].groupby("seed")[metric].mean()
        common_seeds = sorted(set(g1.index) & set(g2.index))
        if len(common_seeds) < 3:
            continue
        a = g1.loc[common_seeds].values
        b = g2.loc[common_seeds].values
        diff = a - b
        _, p_norm = scipy_stats.shapiro(diff) if len(diff) >= 3 else (None, 0.0)
        use_ttest = p_norm > 0.05
        if use_ttest:
            stat, p = scipy_stats.ttest_rel(b, a)  # b vs a: is s2 better than s1?
            effect = cohens_d_paired(b, a)
            test_name = "paired t-test"
        else:
            try:
                stat, p = scipy_stats.wilcoxon(b, a)
            except ValueError:
                stat, p = np.nan, 1.0
            effect = rank_biserial_from_wilcoxon(b, a)
            test_name = "Wilcoxon signed-rank"
        rows.append(dict(comparison=f"{s1} -> {s2}", n_seeds=len(common_seeds),
                          test=test_name, p_value=p, effect_size=effect,
                          effect_label=effect_size_label(effect),
                          mean_s1=a.mean(), mean_s2=b.mean(), delta=b.mean() - a.mean()))

    df = pd.DataFrame(rows)
    if len(df) > 0:
        df["p_adj_holm"] = holm_bonferroni(df["p_value"].values)
        df["significant_adj"] = df["p_adj_holm"] < 0.05
    return df


def friedman_and_nemenyi(results_df, metric="dice", group_col="stage", stage_order=None):
    """Omnibus Friedman test across ALL stages, followed by Nemenyi post-hoc
    if scikit-posthocs is available. Requires the same seeds evaluated on
    every stage (a fully-crossed design)."""
    stages = stage_order or sorted(results_df[group_col].unique())
    pivot = results_df.groupby(["seed", group_col])[metric].mean().unstack(group_col)
    pivot = pivot.dropna()
    pivot = pivot[[s for s in stages if s in pivot.columns]]
    if pivot.shape[0] < 3 or pivot.shape[1] < 3:
        print("  Not enough complete seed x stage combinations for Friedman test yet "
              "(need >=3 seeds fully crossed with >=3 stages).")
        return None, None

    stat, p = scipy_stats.friedmanchisquare(*[pivot[c].values for c in pivot.columns])
    print(f"  Friedman chi^2={stat:.4f}, p={p:.4g} "
          f"({'reject' if p < 0.05 else 'fail to reject'} H0: all stages equal)")

    nemenyi_df = None
    if HAS_POSTHOCS and p < 0.05:
        nemenyi_df = sp.posthoc_nemenyi_friedman(pivot.values)
        nemenyi_df.columns = pivot.columns
        nemenyi_df.index = pivot.columns
        print("  Nemenyi post-hoc p-value matrix:")
        print(nemenyi_df.round(4))
    elif not HAS_POSTHOCS:
        print("  Install scikit-posthocs for Nemenyi post-hoc + critical-difference diagram.")

    return (stat, p), nemenyi_df


def bootstrap_ci(values, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    boot_means = [rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)]
    lo = np.percentile(boot_means, (1 - ci) / 2 * 100)
    hi = np.percentile(boot_means, (1 + ci) / 2 * 100)
    return float(values.mean()), float(lo), float(hi)


def plot_stage_comparison(results_df, metric="dice", stage_order=None, save_path=None):
    stages = stage_order or sorted(results_df["stage"].unique())
    data = [results_df[results_df["stage"] == s][metric].dropna().values for s in stages]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].boxplot(data, labels=stages, showmeans=True)
    axes[0].set_title(f"{metric} by ablation stage (boxplot)")
    axes[0].tick_params(axis='x', rotation=45)

    parts = axes[1].violinplot(data, showmeans=True)
    axes[1].set_xticks(range(1, len(stages) + 1))
    axes[1].set_xticklabels(stages, rotation=45)
    axes[1].set_title(f"{metric} by ablation stage (violin)")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.close()


# ---- smoke test with synthetic per-seed, per-stage results ----
def _smoke_test_stats():
    rng = np.random.default_rng(0)
    rows = []
    stage_means = {"A_baseline": 0.75, "B_plus_x": 0.78, "C_full": 0.83}
    for stage, mu in stage_means.items():
        for seed in [42, 43, 44]:
            for _ in range(5):  # 5 test images per seed
                rows.append(dict(stage=stage, seed=seed, dice=rng.normal(mu, 0.02)))
    df = pd.DataFrame(rows)

    print("Normality report:")
    print(normality_report(df, metric="dice").to_string(index=False))

    print("\nPairwise consecutive-stage comparison (Holm-Bonferroni corrected):")
    pairwise = pairwise_stage_comparison(df, metric="dice",
                                          stage_order=list(stage_means.keys()))
    print(pairwise.to_string(index=False))

    print("\nFriedman omnibus test:")
    friedman_and_nemenyi(df, metric="dice", stage_order=list(stage_means.keys()))

    print("\nBootstrap 95% CI per stage:")
    for stage in stage_means:
        vals = df[df["stage"] == stage]["dice"].values
        mean, lo, hi = bootstrap_ci(vals)
        print(f"  {stage}: {mean:.4f} [{lo:.4f}, {hi:.4f}]")

    assert len(pairwise) == 2, "expected 2 consecutive-stage comparisons for 3 stages"
    assert pairwise["delta"].iloc[-1] > 0, "C_full should outperform B_plus_x on synthetic data"
    print("\nOK: full-model stage correctly detected as improving over intermediate stage.")


print("Running statistics smoke test (synthetic ablation data)...")
_smoke_test_stats()
print("CELL 13 COMPLETE -- statistical analysis verified\n")


Running statistics smoke test (synthetic ablation data)...
Normality report:
     stage  n  shapiro_p  normal
A_baseline 15   0.927028    True
  B_plus_x 15   0.301985    True
    C_full 15   0.718550    True

Pairwise consecutive-stage comparison (Holm-Bonferroni corrected):
            comparison  n_seeds          test  p_value  effect_size effect_label  mean_s1  mean_s2    delta  p_adj_holm  significant_adj
A_baseline -> B_plus_x        3 paired t-test 0.009307     5.942616        large  0.74530 0.779840 0.034540    0.018615             True
    B_plus_x -> C_full        3 paired t-test 0.012624     5.089718        large  0.77984 0.835164 0.055324    0.018615             True

Friedman omnibus test:
  Friedman chi^2=6.0000, p=0.04979 (reject H0: all stages equal)
  Nemenyi post-hoc p-value matrix:
stage       A_baseline  B_plus_x  C_full
stage                                   
A_baseline      1.0000    0.4385  0.0380
B_plus_x        0.4385    1.0000  0.4385
C_full          0.0380  

In [13]:
# ============================================================
# CELL 14: ROBUSTNESS ANALYSIS UNDER IMAGE CORRUPTIONS
# ============================================================
# Applies corruptions to the ALREADY-preprocessed (CLAHE) image before
# overlap-tile inference, at multiple severities, and tracks the Dice
# degradation curve per corruption type. NEEDS REAL DATA + a trained model.

def add_gaussian_noise(img, severity):
    sigma = [5, 10, 20, 35, 50][severity - 1]
    noise = np.random.normal(0, sigma, img.shape)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def add_speckle_noise(img, severity):
    var = [0.01, 0.03, 0.06, 0.1, 0.15][severity - 1]
    noise = np.random.normal(0, var ** 0.5, img.shape)
    return np.clip(img.astype(np.float32) * (1 + noise), 0, 255).astype(np.uint8)


def add_salt_pepper(img, severity):
    amount = [0.005, 0.01, 0.02, 0.04, 0.08][severity - 1]
    out = img.copy()
    mask = np.random.random(img.shape) < amount
    salt = np.random.random(img.shape) < 0.5
    out[mask & salt] = 255
    out[mask & ~salt] = 0
    return out


def add_motion_blur(img, severity):
    ksize = [3, 5, 9, 13, 17][severity - 1]
    kernel = np.zeros((ksize, ksize))
    kernel[ksize // 2, :] = 1.0 / ksize
    return cv2.filter2D(img, -1, kernel)


def reduce_contrast(img, severity):
    factor = [0.9, 0.75, 0.6, 0.45, 0.3][severity - 1]
    mean = img.mean()
    return np.clip((img.astype(np.float32) - mean) * factor + mean, 0, 255).astype(np.uint8)


def add_illumination_variation(img, severity):
    strength = [0.1, 0.2, 0.35, 0.5, 0.7][severity - 1]
    h, w = img.shape
    yy, xx = np.mgrid[0:h, 0:w]
    gradient = (xx / w) * strength * 255
    return np.clip(img.astype(np.float32) + gradient - gradient.mean(), 0, 255).astype(np.uint8)


def add_jpeg_compression(img, severity):
    quality = [80, 60, 40, 20, 10][severity - 1]
    ok, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return cv2.imdecode(enc, cv2.IMREAD_GRAYSCALE)


def add_partial_occlusion(img, severity):
    frac = [0.02, 0.05, 0.08, 0.12, 0.18][severity - 1]
    out = img.copy()
    h, w = img.shape
    n_patches = 3
    patch_area = int(frac * h * w / n_patches)
    side = int(patch_area ** 0.5)
    for _ in range(n_patches):
        y = np.random.randint(0, max(1, h - side))
        x = np.random.randint(0, max(1, w - side))
        out[y:y + side, x:x + side] = 0
    return out


CORRUPTIONS = {
    "gaussian_noise": add_gaussian_noise,
    "speckle_noise": add_speckle_noise,
    "salt_pepper": add_salt_pepper,
    "motion_blur": add_motion_blur,
    "low_contrast": reduce_contrast,
    "illumination": add_illumination_variation,
    "jpeg_compression": add_jpeg_compression,
    "partial_occlusion": add_partial_occlusion,
}


def robustness_evaluation(model, image_paths, mask_paths, precomputed_dir, cfg,
                           severities=(1, 2, 3, 4, 5)):
    rows = []
    for corruption_name, corrupt_fn in CORRUPTIONS.items():
        for severity in severities:
            for ip, mp in zip(image_paths, mask_paths):
                base = os.path.splitext(os.path.basename(mp))[0]
                npz_path = os.path.join(precomputed_dir, f"{base}_reps.npz")
                if not os.path.exists(npz_path):
                    continue

                image = np.array(Image.open(ip).convert('RGB'))
                green = image[:, :, 1]
                clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
                preprocessed = clahe.apply(green)

                # Align GT mask to the image's actual shape, NOT the mask
                # file's native shape (they can differ -- see _align_to_shape
                # in Cell 11, same fix applied here).
                raw_mask = dict(np.load(npz_path))['mask'].astype(np.float32)
                gt_mask = _align_to_shape(raw_mask, preprocessed.shape[:2], is_binary=True)

                corrupted = corrupt_fn(preprocessed, severity)
                pred = overlap_tile_predict(model, corrupted, patch_size=cfg.PATCH_SIZE,
                                             stride=int(cfg.PATCH_SIZE * 0.75))
                pred_mask = (pred['mask'] > 0.5).astype(np.float32)
                d = dice_score(pred_mask, gt_mask)
                rows.append(dict(corruption=corruption_name, severity=severity,
                                  image=os.path.basename(ip), dice=d))

    return pd.DataFrame(rows)


def plot_degradation_curves(robustness_df, save_path=None):
    fig, ax = plt.subplots(figsize=(9, 6))
    for corruption, g in robustness_df.groupby("corruption"):
        agg = g.groupby("severity")["dice"].mean()
        ax.plot(agg.index, agg.values, marker='o', label=corruption)
    ax.set_xlabel("Severity")
    ax.set_ylabel("Dice")
    ax.set_title("Robustness degradation curves")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.close()


# ---- smoke test: corruption functions produce valid images at every severity ----
def _smoke_test_corruptions():
    img = (np.random.rand(96, 96) * 255).astype(np.uint8)
    for name, fn in CORRUPTIONS.items():
        for sev in range(1, 6):
            out = fn(img, sev)
            assert out.shape == img.shape, f"{name} sev{sev}: shape changed to {out.shape}"
            assert out.dtype == np.uint8, f"{name} sev{sev}: dtype {out.dtype}"
        print(f"  {name}: all 5 severities OK")


print("Running corruption-function smoke test...")
_smoke_test_corruptions()
print("CELL 14 COMPLETE -- robustness_evaluation() ready (needs real data + trained model)\n")


Running corruption-function smoke test...
  gaussian_noise: all 5 severities OK
  speckle_noise: all 5 severities OK
  salt_pepper: all 5 severities OK
  motion_blur: all 5 severities OK
  low_contrast: all 5 severities OK
  illumination: all 5 severities OK
  jpeg_compression: all 5 severities OK
  partial_occlusion: all 5 severities OK
CELL 14 COMPLETE -- robustness_evaluation() ready (needs real data + trained model)



In [14]:
# ============================================================
# CELL 15: EXPLAINABILITY (XAI)
# ============================================================
# Grad-CAM via the pytorch_grad_cam library (already in the original
# notebook's dependency list), plus two methods implemented directly
# (no extra dependency): Integrated Gradients and Occlusion Sensitivity.
# A quantitative "pointing game" score checks whether each method's peak
# activation lands inside the ground-truth junction map -- i.e. whether
# the explanation actually points at anatomically meaningful structure,
# not just "somewhere on a vessel."

class MaskOutputWrapper(torch.nn.Module):
    """Grad-CAM/IG need a model that returns a single tensor, not a dict."""
    def __init__(self, model, output_key="mask"):
        super().__init__()
        self.model = model
        self.output_key = output_key

    def forward(self, x):
        return self.model(x)[self.output_key]


def integrated_gradients(model, input_tensor, target_key="mask", steps=30, baseline=None):
    """IG w.r.t. the SUM of predicted mask logits inside the vessel region.
    No external dependency (captum optional but not required)."""
    model.eval()
    if baseline is None:
        baseline = torch.zeros_like(input_tensor)

    scaled_inputs = [baseline + (float(i) / steps) * (input_tensor - baseline) for i in range(steps + 1)]
    grads = []
    for scaled in scaled_inputs:
        scaled = scaled.clone().detach().requires_grad_(True)
        out = model(scaled)[target_key]
        score = torch.sigmoid(out).sum()
        grad = torch.autograd.grad(score, scaled)[0]
        grads.append(grad.detach())

    avg_grad = torch.stack(grads).mean(dim=0)
    ig = (input_tensor - baseline) * avg_grad
    return ig.sum(dim=1, keepdim=True)  # (B, 1, H, W) attribution map


def occlusion_sensitivity(model, input_tensor, target_key="mask", patch_size=16, stride=8):
    """Slides an occluding patch over the image, measures drop in predicted
    vessel probability -- importance = how much the prediction depended on
    that region."""
    model.eval()
    with torch.no_grad():
        base_out = torch.sigmoid(model(input_tensor)[target_key])
        base_score = base_out.sum().item()

    _, _, H, W = input_tensor.shape
    heatmap = torch.zeros(1, 1, H, W)
    counts = torch.zeros(1, 1, H, W)

    with torch.no_grad():
        for y in range(0, H - patch_size + 1, stride):
            for x in range(0, W - patch_size + 1, stride):
                occluded = input_tensor.clone()
                occluded[:, :, y:y + patch_size, x:x + patch_size] = 0
                out = torch.sigmoid(model(occluded)[target_key])
                drop = base_score - out.sum().item()
                heatmap[:, :, y:y + patch_size, x:x + patch_size] += drop
                counts[:, :, y:y + patch_size, x:x + patch_size] += 1

    return heatmap / counts.clamp(min=1)


def pointing_game_score(attribution_map, junction_map, radius=5):
    """Fraction of the attribution map's peak location that falls within
    `radius` px of a ground-truth junction pixel -- a quantitative check
    that an explanation method is actually keying on anatomically important
    structure (bifurcations), not just any vessel pixel."""
    attr = attribution_map.squeeze().detach().cpu().numpy()
    peak_y, peak_x = np.unravel_index(np.argmax(attr), attr.shape)
    junction = junction_map.squeeze().detach().cpu().numpy() if torch.is_tensor(junction_map) else junction_map
    ys, xs = np.where(junction > 0)
    if len(ys) == 0:
        return np.nan
    dists = np.sqrt((ys - peak_y) ** 2 + (xs - peak_x) ** 2)
    return float(dists.min() <= radius)


def compare_xai_methods(model, input_tensor, junction_map, target_layer=None, save_path=None):
    """Runs Grad-CAM (if available), Integrated Gradients, and Occlusion
    Sensitivity on one input patch and produces a side-by-side figure plus
    pointing-game scores."""
    results = {}

    wrapped = MaskOutputWrapper(model, "mask")
    if HAS_GRADCAM and target_layer is not None:
        cam = GradCAM(model=wrapped, target_layers=[target_layer])
        grayscale_cam = cam(input_tensor=input_tensor,
                             targets=[SemanticSegmentationTarget(0, np.ones(input_tensor.shape[-2:]))])
        results["Grad-CAM"] = torch.from_numpy(grayscale_cam[0]).unsqueeze(0).unsqueeze(0)

    results["Integrated Gradients"] = integrated_gradients(model, input_tensor).abs()
    results["Occlusion Sensitivity"] = occlusion_sensitivity(model, input_tensor)

    fig, axes = plt.subplots(1, len(results) + 1, figsize=(4 * (len(results) + 1), 4))
    img_np = input_tensor[0].permute(1, 2, 0).detach().cpu().numpy()
    axes[0].imshow(img_np)
    axes[0].set_title("Input")
    axes[0].axis('off')

    scores = {}
    for ax, (name, attr) in zip(axes[1:], results.items()):
        attr_np = attr.squeeze().detach().cpu().numpy()
        ax.imshow(img_np)
        ax.imshow(attr_np, cmap='jet', alpha=0.5)
        score = pointing_game_score(attr, junction_map)
        scores[name] = score
        ax.set_title(f"{name}\npointing-game={score:.0f}" if not np.isnan(score) else name)
        ax.axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.close()
    return scores


# ---- smoke test with a tiny random model (no real dataset needed) ----
def _smoke_test_xai():
    torch.manual_seed(0)
    m = TopoEyeNetV2(encoder_weights=None, use_skeleton_head=True, use_btgi=True, use_jaa=True)
    m.eval()
    x = torch.randn(1, 3, 64, 64, requires_grad=False)
    junction = np.zeros((64, 64), dtype=np.uint8)
    junction[30:34, 30:34] = 1

    ig = integrated_gradients(m, x, steps=5)
    assert ig.shape == (1, 1, 64, 64), ig.shape
    occ = occlusion_sensitivity(m, x, patch_size=16, stride=16)
    assert occ.shape == (1, 1, 64, 64), occ.shape
    score = pointing_game_score(ig, junction)
    print(f"  IG shape OK, Occlusion shape OK, pointing-game score example={score}")


print("Running XAI smoke test (random model, no dataset needed)...")
_smoke_test_xai()
print("CELL 15 COMPLETE -- XAI pipeline verified (Grad-CAM needs a target_layer + real image at use time)\n")


Running XAI smoke test (random model, no dataset needed)...
  IG shape OK, Occlusion shape OK, pointing-game score example=1.0
CELL 15 COMPLETE -- XAI pipeline verified (Grad-CAM needs a target_layer + real image at use time)



In [15]:
# ============================================================
# CELL 16: BASELINE COMPARISON (U-Net, UNet++, Attention U-Net, DeepLabV3+)
# ============================================================
# Uses segmentation_models_pytorch so every baseline shares the SAME
# encoder, preprocessing, patch pipeline, and optimizer as TopoEyeNetV2 --
# only the architecture differs, which is what a fair IEEE Access
# comparison table requires. NEEDS REAL DATA + GPU to actually train.

BASELINE_BUILDERS = {
    "UNet": lambda enc: smp.Unet(encoder_name=enc, encoder_weights="imagenet", in_channels=3, classes=1),
    "UNet++": lambda enc: smp.UnetPlusPlus(encoder_name=enc, encoder_weights="imagenet", in_channels=3, classes=1),
    "DeepLabV3+": lambda enc: smp.DeepLabV3Plus(encoder_name=enc, encoder_weights="imagenet", in_channels=3, classes=1),
    "FPN": lambda enc: smp.FPN(encoder_name=enc, encoder_weights="imagenet", in_channels=3, classes=1),
    # Attention U-Net, SA-UNet, FR-UNet, LadderNet, IterNet, CE-Net, DUNet are not
    # in segmentation_models_pytorch and would need their own implementations or
    # third-party repos vendored in -- flagged here rather than silently
    # skipped so the comparison table doesn't imply they were run when they weren't.
}


def train_baseline(name, builder_fn, train_loader, val_loader, cfg, seed=42):
    seed_everything(seed)
    model = builder_fn(cfg.ENCODER).to(device)
    model = wrap_dual_gpu(model)
    criterion = DiceLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS, eta_min=cfg.ETA_MIN)
    scaler = GradScaler(enabled=cfg.USE_AMP)

    ckpt_path = os.path.join(cfg.CHECKPOINT_DIR, f"baseline_{name}_seed{seed}.pt")
    best_val_dice = -1.0  # see note in Cell 10 -- must not equal a real achievable val_dice
    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        for images, targets in train_loader:
            images = images.to(device, non_blocking=True)
            masks = targets['mask'].to(device, non_blocking=True)
            optimizer.zero_grad()
            with autocast(enabled=cfg.USE_AMP):
                logits = model(images)
                loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()

        model.eval()
        val_dice, n = 0.0, 0
        with torch.no_grad():
            for images, targets in val_loader:
                images = images.to(device, non_blocking=True)
                masks = targets['mask'].to(device, non_blocking=True)
                with autocast(enabled=cfg.USE_AMP):
                    logits = model(images)
                val_dice += dice_from_logits(logits, masks)
                n += 1
        val_dice /= max(1, n)
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            torch.save(state, ckpt_path)
        if epoch % 10 == 0 or epoch == cfg.EPOCHS:
            print(f"  [{name}] epoch {epoch}/{cfg.EPOCHS} | val_dice={val_dice:.4f} (best={best_val_dice:.4f})")
        clear_gpu_cache()

    return best_val_dice, ckpt_path


def run_baseline_comparison(cfg, train_loader, val_loader, test_images, test_masks,
                             precomputed_dir, seeds=None):
    seeds = seeds or cfg.SEEDS
    all_results = []
    for name, builder in BASELINE_BUILDERS.items():
        for seed in seeds:
            print(f"\nTraining baseline: {name} (seed={seed})")
            try:
                _, ckpt_path = train_baseline(name, builder, train_loader, val_loader, cfg, seed=seed)
                if not os.path.exists(ckpt_path):
                    print(f"  WARNING: no checkpoint saved for baseline {name} seed={seed} -- skipping.")
                    continue
                model = builder(cfg.ENCODER).to(device)
                model.load_state_dict(torch.load(ckpt_path, map_location=device))

                model_wrapped = _SingleHeadWrapper(model)
                test_df = evaluate_model_on_set(model_wrapped, test_images, test_masks, precomputed_dir,
                                                 patch_size=cfg.PATCH_SIZE, stride=int(cfg.PATCH_SIZE * 0.75))
                test_df['stage'] = f"baseline_{name}"
                test_df['seed'] = seed
                test_df['n_params'] = sum(p.numel() for p in model.parameters())
                all_results.append(test_df)
                del model
                clear_gpu_cache()
            except Exception as e:
                # A single baseline failing (e.g. a transient pretrained-weight
                # download error) should not take down the rest of the
                # comparison table -- log it and move on to the next baseline.
                print(f"  ERROR training baseline {name} seed={seed}: {e!r} -- skipping this run.")
                continue

    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()


class _SingleHeadWrapper(torch.nn.Module):
    """Wraps a plain smp model (returns a raw tensor) so it matches the
    dict-output interface evaluate_model_on_set()/overlap_tile_predict()
    expect from TopoEyeNetV2."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return {"mask": self.model(x)}


print("CELL 16 COMPLETE -- run_baseline_comparison() ready. "
      "Note: SA-UNet/FR-UNet/LadderNet/IterNet/CE-Net/DUNet need separate "
      "implementations not included here -- see the note in BASELINE_BUILDERS.")


CELL 16 COMPLETE -- run_baseline_comparison() ready. Note: SA-UNet/FR-UNet/LadderNet/IterNet/CE-Net/DUNet need separate implementations not included here -- see the note in BASELINE_BUILDERS.


In [16]:
# ============================================================
# CELL 17: COMPUTATIONAL COST ANALYSIS
# ============================================================
# Params, FLOPs (via ptflops if available, else a manual conv-FLOP counter
# fallback so this cell never hard-fails), FPS, GPU memory, model size.

def count_flops_manual(model, input_size=(1, 3, 128, 128)):
    """Fallback FLOP counter: sums 2*Cin*Cout*K*K*Hout*Wout for every Conv2d
    via forward hooks. Rough but dependency-free and good enough for a
    relative (not absolute) comparison across ablation stages."""
    flops = [0]
    hooks = []

    def hook(module, inp, out):
        if isinstance(module, torch.nn.Conv2d):
            out_h, out_w = out.shape[-2:]
            k_h, k_w = module.kernel_size
            flops[0] += 2 * module.in_channels * module.out_channels * k_h * k_w * out_h * out_w / module.groups

    for m in model.modules():
        if isinstance(m, torch.nn.Conv2d):
            hooks.append(m.register_forward_hook(hook))

    model.eval()
    with torch.no_grad():
        model(torch.randn(*input_size).to(next(model.parameters()).device))
    for h in hooks:
        h.remove()
    return flops[0]


def measure_fps(model, input_size=(1, 3, 128, 128), n_warmup=10, n_iters=50):
    model.eval()
    x = torch.randn(*input_size).to(next(model.parameters()).device)
    with torch.no_grad():
        for _ in range(n_warmup):
            model(x)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_iters):
            model(x)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.time() - t0
    return n_iters / elapsed


def computational_profile(model, input_size=(1, 3, 128, 128)):
    n_params = sum(p.numel() for p in model.parameters())
    model_size_mb = n_params * 4 / 1e6  # float32
    flops = count_flops_manual(model, input_size)
    fps = measure_fps(model, input_size)

    mem_mb = None
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        model(torch.randn(*input_size).to(next(model.parameters()).device))
        mem_mb = torch.cuda.max_memory_allocated() / 1e6

    return dict(n_params=n_params, model_size_mb=model_size_mb,
                gflops=flops / 1e9, fps=fps, peak_mem_mb=mem_mb)


def compare_computational_cost(model_configs, cfg, input_size=(1, 3, 128, 128)):
    """model_configs: dict[name] -> nn.Module instance (already built)."""
    rows = []
    for name, model in model_configs.items():
        model = model.to(device)
        prof = computational_profile(model, input_size)
        prof['model'] = name
        rows.append(prof)
        del model
        clear_gpu_cache()
    df = pd.DataFrame(rows).set_index('model')
    return df[['n_params', 'model_size_mb', 'gflops', 'fps', 'peak_mem_mb']]


# ---- smoke test on CPU with tiny inputs ----
def _smoke_test_compute_profile():
    m = TopoEyeNetV2(encoder_weights=None, use_skeleton_head=True, use_btgi=True, use_jaa=True)
    m = m.to(device)
    prof = computational_profile(m, input_size=(1, 3, 64, 64))
    print(f"  params={prof['n_params']:,}  size={prof['model_size_mb']:.1f}MB  "
          f"GFLOPs={prof['gflops']:.3f}  FPS={prof['fps']:.1f}")
    assert prof['n_params'] > 0 and prof['gflops'] > 0


print("Running computational-profile smoke test...")
_smoke_test_compute_profile()
print("CELL 17 COMPLETE -- compare_computational_cost() ready\n")


Running computational-profile smoke test...
  params=27,626,403  size=110.5MB  GFLOPs=1.438  FPS=132.8
CELL 17 COMPLETE -- compare_computational_cost() ready



In [17]:
# ============================================================
# CELL 18: FINAL RESULTS AGGREGATION + LATEX TABLE
# ============================================================

def make_latex_table(summary_df, caption="Ablation study results", label="tab:ablation"):
    """summary_df: MultiIndex columns (metric, stat) as produced by
    summarize_ablation(); renders mean +/- std per cell, IEEE-Access style."""
    metrics = summary_df.columns.get_level_values(0).unique()
    lines = []
    lines.append("\\begin{table}[t]")
    lines.append(f"\\caption{{{caption}}}")
    lines.append(f"\\label{{{label}}}")
    lines.append("\\centering")
    lines.append("\\begin{tabular}{l" + "c" * len(metrics) + "}")
    lines.append("\\hline")
    lines.append("Stage & " + " & ".join(str(m) for m in metrics) + " \\\\")
    lines.append("\\hline")
    for stage in summary_df.index:
        cells = []
        for m in metrics:
            mean = summary_df.loc[stage, (m, "mean")]
            std = summary_df.loc[stage, (m, "std")]
            cells.append(f"{mean:.3f}$\\pm${std:.3f}" if pd.notna(std) else f"{mean:.3f}")
        stage_label = str(stage).replace("_", "\\_")
        lines.append(f"{stage_label} & " + " & ".join(cells) + " \\\\")
    lines.append("\\hline")
    lines.append("\\end{tabular}")
    lines.append("\\end{table}")
    return "\n".join(lines)


def generate_final_report(ablation_results_df, stat_pairwise_df, compute_df, cfg,
                           metrics=("dice", "iou", "clDice", "betti0_error", "connectivity")):
    """Prints + saves a markdown summary tying together ablation results,
    statistical tests, and computational cost -- the pieces a paper's
    results section needs, assembled automatically from what actually ran."""
    summary = summarize_ablation(ablation_results_df, metrics=metrics)

    report_lines = ["# TopoEyeNet v2 -- Results Summary\n"]
    report_lines.append("## Ablation study (mean +/- std across seeds)\n")
    report_lines.append(summary.to_markdown() if hasattr(summary, "to_markdown") else str(summary))

    report_lines.append("\n## Statistically significant improvements (Holm-Bonferroni corrected, alpha=0.05)\n")
    if stat_pairwise_df is not None and len(stat_pairwise_df) > 0:
        sig = stat_pairwise_df[stat_pairwise_df["significant_adj"]]
        if len(sig) > 0:
            report_lines.append(sig[["comparison", "test", "p_adj_holm", "effect_size", "effect_label", "delta"]]
                                 .to_markdown(index=False))
        else:
            report_lines.append("No stage-to-stage improvement survived correction at this sample size.")

    if compute_df is not None and len(compute_df) > 0:
        report_lines.append("\n## Computational cost\n")
        report_lines.append(compute_df.to_markdown())

    report_lines.append("\n## LaTeX table (paste into paper)\n```latex")
    report_lines.append(make_latex_table(summary))
    report_lines.append("```")

    report = "\n".join(report_lines)
    report_path = os.path.join(cfg.LOG_DIR, "final_report.md")
    with open(report_path, "w") as f:
        f.write(report)
    print(f"Saved: {report_path}")
    print(report)
    return report


print("CELL 18 COMPLETE -- generate_final_report(ablation_results_df, stat_pairwise_df, "
      "compute_df, cfg) ready")


CELL 18 COMPLETE -- generate_final_report(ablation_results_df, stat_pairwise_df, compute_df, cfg) ready


In [ ]:
# ============================================================
# CELL 19: RUN EVERYTHING (this is what actually produces results)
# ============================================================
# Every prior cell only DEFINED functions. This cell CALLS them, in order,
# and is the one that actually trains models and produces the final
# tables/figures. If you only wanted to inspect the code, stop before this
# cell. If you want results, run this.
#
# Guardrail: refuses to run if Cell 3 found no data, so you get a clear
# message instead of a confusing crash three functions deep.

if len(rm_train_images) == 0:
    raise RuntimeError(
        "No training data found (Cell 3 discovered 0 RetinoMix-5 pairs). "
        "Check cfg.RETINOMIX_IMAGE_DIR / cfg.RETINOMIX_MASK_DIR point at the "
        "attached Kaggle dataset before running this cell."
    )

print("=" * 70)
print("STEP 1/6: Building patch DataLoaders")
print("=" * 70)
patch_train_loader, patch_val_loader = build_patch_loaders(
    list(rm_train_images), list(rm_train_masks),
    list(rm_val_images), list(rm_val_masks),
    RM_PRECOMPUTED_DIR, cfg,
)

print("\n" + "=" * 70)
print("STEP 2/6: Progressive ablation study (all 6 stages x all seeds)")
print("=" * 70)
ablation_results_df = run_ablation_study(
    cfg, patch_train_loader, patch_val_loader,
    list(rm_test_images), list(rm_test_masks), RM_PRECOMPUTED_DIR,
)
print("\nAblation summary:")
ablation_summary = summarize_ablation(ablation_results_df)
print(ablation_summary)

print("\n" + "=" * 70)
print("STEP 3/6: Statistical significance analysis")
print("=" * 70)
stage_order = list(ABLATION_STAGES.keys())
print("\nNormality report (Dice):")
print(normality_report(ablation_results_df, metric="dice"))
print("\nPairwise consecutive-stage comparison:")
stat_pairwise_df = pairwise_stage_comparison(ablation_results_df, metric="dice", stage_order=stage_order)
print(stat_pairwise_df)
print("\nFriedman + Nemenyi omnibus test:")
friedman_result, nemenyi_df = friedman_and_nemenyi(ablation_results_df, metric="dice", stage_order=stage_order)
plot_stage_comparison(ablation_results_df, metric="dice", stage_order=stage_order,
                       save_path=os.path.join(cfg.FIGURES_DIR, "ablation_dice_comparison.png"))

print("\n" + "=" * 70)
print("STEP 4/6: Robustness under corruption (best-stage model only, all seeds)")
print("=" * 70)
best_stage_name = list(ABLATION_STAGES.keys())[-1]  # full model
best_seed = cfg.SEEDS[0]
best_ckpt_path = os.path.join(cfg.CHECKPOINT_DIR, f"{best_stage_name}_seed{best_seed}.pt")
if os.path.exists(best_ckpt_path):
    best_model = TopoEyeNetV2(
        encoder_name=cfg.ENCODER, encoder_weights=None,
        in_channels=cfg.IN_CHANNELS, decoder_channels=cfg.DECODER_CHANNELS,
        btgi_stages=cfg.BTGI_STAGES, jaa_lambda=cfg.JAA_LAMBDA,
        **ABLATION_STAGES[best_stage_name]["model_kwargs"],
    ).to(device)
    ckpt = torch.load(best_ckpt_path, map_location=device)
    best_model.load_state_dict(ckpt['state_dict'])
    robustness_df = robustness_evaluation(best_model, list(rm_test_images), list(rm_test_masks),
                                           RM_PRECOMPUTED_DIR, cfg)
    robustness_df.to_csv(os.path.join(cfg.LOG_DIR, "robustness_results.csv"), index=False)
    plot_degradation_curves(robustness_df, save_path=os.path.join(cfg.FIGURES_DIR, "robustness_curves.png"))
    print(robustness_df.groupby("corruption")["dice"].mean().sort_values())
else:
    print(f"Skipping -- no checkpoint found at {best_ckpt_path}")
    robustness_df = pd.DataFrame()

print("\n" + "=" * 70)
print("STEP 5/6: Baseline comparison (U-Net, UNet++, DeepLabV3+, FPN)")
print("=" * 70)
baseline_results_df = run_baseline_comparison(
    cfg, patch_train_loader, patch_val_loader,
    list(rm_test_images), list(rm_test_masks), RM_PRECOMPUTED_DIR,
)
if len(baseline_results_df) > 0:
    combined_df = pd.concat([ablation_results_df, baseline_results_df], ignore_index=True)
else:
    combined_df = ablation_results_df

print("\n" + "=" * 70)
print("STEP 6/6: Computational cost + final report")
print("=" * 70)
compute_configs = {}
for stage_name, stage_cfg in ABLATION_STAGES.items():
    compute_configs[stage_name] = TopoEyeNetV2(
        encoder_name=cfg.ENCODER, encoder_weights=None,
        in_channels=cfg.IN_CHANNELS, decoder_channels=cfg.DECODER_CHANNELS,
        btgi_stages=cfg.BTGI_STAGES, jaa_lambda=cfg.JAA_LAMBDA,
        **stage_cfg["model_kwargs"],
    )
compute_df = compare_computational_cost(compute_configs, cfg, input_size=(1, cfg.IN_CHANNELS, cfg.PATCH_SIZE, cfg.PATCH_SIZE))
print(compute_df)

final_report = generate_final_report(combined_df, stat_pairwise_df, compute_df, cfg)

print("\n" + "=" * 70)
print("ALL DONE. Outputs written to:")
print(f"  {cfg.LOG_DIR}/ablation_full_results.csv")
print(f"  {cfg.LOG_DIR}/robustness_results.csv")
print(f"  {cfg.LOG_DIR}/final_report.md")
print(f"  {cfg.FIGURES_DIR}/*.png")
print("=" * 70)


STEP 1/6: Building patch DataLoaders
TopoPatchDataset: 163 source images, 60 patches/image = 9780 patches/epoch
    cached 20/163 (4.7s elapsed)
    cached 40/163 (7.5s elapsed)
    cached 60/163 (13.0s elapsed)
    cached 80/163 (16.4s elapsed)
    cached 100/163 (20.5s elapsed)
    cached 120/163 (23.3s elapsed)
    cached 140/163 (26.3s elapsed)
    cached 160/163 (30.0s elapsed)
    cached 163/163 (30.9s elapsed)
TopoPatchDataset: 35 source images, 20 patches/image = 700 patches/epoch
    cached 20/35 (1.4s elapsed)
    cached 35/35 (3.5s elapsed)

STEP 2/6: Progressive ablation study (all 6 stages x all seeds)

ABLATION STAGE: A_baseline_mask_only | seed=42


Downloading: "https://download.pytorch.org/models/resnet34-333f7ec4.pth" to /root/.cache/torch/hub/checkpoints/resnet34-333f7ec4.pth
100%|██████████| 83.3M/83.3M [00:00<00:00, 274MB/s]


  [A_baseline_mask_only seed=42] epoch 1/30 | total=0.3947 | val_dice=0.7142 (best=0.7142) | 149.0s
  [A_baseline_mask_only seed=42] epoch 5/30 | total=0.2491 | val_dice=0.7673 (best=0.7673) | 166.6s
  [A_baseline_mask_only seed=42] epoch 10/30 | total=0.2261 | val_dice=0.7889 (best=0.7889) | 153.3s
  [A_baseline_mask_only seed=42] epoch 15/30 | total=0.2185 | val_dice=0.7987 (best=0.7987) | 147.8s
  [A_baseline_mask_only seed=42] epoch 20/30 | total=0.2137 | val_dice=0.8057 (best=0.8057) | 145.1s
  [A_baseline_mask_only seed=42] epoch 25/30 | total=0.2109 | val_dice=0.8045 (best=0.8058) | 145.0s
  [A_baseline_mask_only seed=42] epoch 30/30 | total=0.2106 | val_dice=0.8051 (best=0.8109) | 149.3s

ABLATION STAGE: A_baseline_mask_only | seed=43
  [A_baseline_mask_only seed=43] epoch 1/30 | total=0.3774 | val_dice=0.7292 (best=0.7292) | 147.8s
  [A_baseline_mask_only seed=43] epoch 5/30 | total=0.2476 | val_dice=0.6906 (best=0.7612) | 144.3s
  [A_baseline_mask_only seed=43] epoch 10/30 | 